In [ ]:
import cv2
import numpy as np
import glob
from pathlib import Path

Part 0:

In [ ]:
ARUCO_TAG_SIZE = 0.06  # Size of your ArUco tag in meters (adjust to your actual size)
CALIBRATION_IMAGES_PATH = "aruco/*.jpg"  # Adjust path and extension as needed

# Create ArUco dictionary and detector (NEW API for OpenCV 4.7+)
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
aruco_params = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)

# Define 3D coordinates of ArUco tag corners in world space
objp = np.array([
    [0, 0, 0],
    [ARUCO_TAG_SIZE, 0, 0],
    [ARUCO_TAG_SIZE, ARUCO_TAG_SIZE, 0],
    [0, ARUCO_TAG_SIZE, 0]
], dtype=np.float32)

# Storage for calibration data
all_object_points = []  # 3D points in world coordinates
all_image_points = []   # 2D points in image coordinates
image_size = None

# Load calibration images
image_paths = glob.glob(CALIBRATION_IMAGES_PATH)
print(f"Found {len(image_paths)} calibration images")

successful_detections = 0

# Process each calibration image
for img_path in image_paths:
    # Read image
    image = cv2.imread(img_path)
    
    if image is None:
        print(f"Warning: Could not read image {img_path}")
        continue
    
    # Store image size (needed for calibration)
    if image_size is None:
        image_size = (image.shape[1], image.shape[0])  # (width, height)
    
    # Convert to grayscale for ArUco detection
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Detect ArUco markers (NEW API)
    corners, ids, rejected = detector.detectMarkers(gray)
    
    # Check if any markers were detected
    if ids is not None and len(ids) > 0:
        print(f"✓ {Path(img_path).name}: Detected {len(ids)} tag(s)")
        
        # Process each detected tag
        for i, corner in enumerate(corners):
            # corner shape: (1, 4, 2) -> reshape to (4, 2)
            corner_points = corner.reshape(4, 2).astype(np.float32)
            
            # Add to calibration data
            all_object_points.append(objp)
            all_image_points.append(corner_points)
            
        successful_detections += 1
    else:
        print(f"✗ {Path(img_path).name}: No tags detected")

print(f"\nSuccessfully detected tags in {successful_detections}/{len(image_paths)} images")
print(f"Total tag detections: {len(all_object_points)}")

# Perform camera calibration
if len(all_object_points) >= 10:  # Need at least ~10 detections for good calibration
    print("\nPerforming camera calibration...")
    
    ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        all_object_points,
        all_image_points,
        image_size,
        None,
        None
    )
    
    print(f"\nCalibration successful!")
    print(f"Reprojection error: {ret:.4f} pixels")
    print(f"\nCamera Matrix (K):")
    print(camera_matrix)
    print(f"\nDistortion Coefficients:")
    print(dist_coeffs)
    
    # Save calibration results
    np.savez('camera_calibration.npz',
             camera_matrix=camera_matrix,
             dist_coeffs=dist_coeffs,
             rvecs=rvecs,
             tvecs=tvecs,
             reprojection_error=ret)
    print("\nCalibration data saved to 'camera_calibration.npz'")
    
else:
    print(f"\nError: Not enough detections for calibration (found {len(all_object_points)}, need at least 10)")
    print("Try capturing more images or adjusting lighting conditions")


In [ ]:
import viser
import time

In [ ]:
# Load your camera calibration
calib_data = np.load('camera_calibration.npz')
camera_matrix = calib_data['camera_matrix']
dist_coeffs = calib_data['dist_coeffs']

print("Camera calibration loaded successfully")
print(f"Camera Matrix:\n{camera_matrix}")

# Configuration
ARUCO_TAG_SIZE = 0.06  
OBJECT_IMAGES_PATH = "nerf/*.jpg"  # Path to your object scan images

# Create ArUco detector
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
aruco_params = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)

# Define 3D coordinates of ArUco tag corners in world space
# Same as calibration - tag at origin on z=0 plane
object_points = np.array([
    [0, 0, 0],
    [ARUCO_TAG_SIZE, 0, 0],
    [ARUCO_TAG_SIZE, ARUCO_TAG_SIZE, 0],
    [0, ARUCO_TAG_SIZE, 0]
], dtype=np.float32)

# Load object images
image_paths = sorted(glob.glob(OBJECT_IMAGES_PATH))
print(f"\nFound {len(image_paths)} object images")

# Storage for camera poses
camera_poses = []  # List of dictionaries containing pose info
successful_poses = 0

# Process each image
for i, img_path in enumerate(image_paths):
    # Read image
    image = cv2.imread(img_path)
    
    if image is None:
        print(f"Warning: Could not read image {img_path}")
        continue
    
    H, W = image.shape[:2]
    
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Detect ArUco markers
    corners, ids, rejected = detector.detectMarkers(gray)
    
    # Check if exactly one marker was detected
    if ids is not None and len(ids) > 0:
        # Use the first detected marker
        image_points = corners[0].reshape(4, 2).astype(np.float32)
        
        # Solve PnP to get camera pose
        success, rvec, tvec = cv2.solvePnP(
            object_points,
            image_points,
            camera_matrix,
            dist_coeffs
        )
        
        if success:
            # Convert rotation vector to rotation matrix
            R, _ = cv2.Rodrigues(rvec)
            
            # solvePnP returns world-to-camera transformation
            # We need camera-to-world (c2w) for visualization
            # c2w = [R | t]^(-1)
            
            # Invert the transformation
            R_inv = R.T
            t_inv = -R_inv @ tvec
            
            # Create 4x4 camera-to-world matrix
            c2w = np.eye(4)
            c2w[:3, :3] = R_inv
            c2w[:3, 3] = t_inv.flatten()
            
            # Store pose information
            camera_poses.append({
                'index': i,
                'image_path': img_path,
                'image': image,
                'c2w': c2w,
                'width': W,
                'height': H,
                'rvec': rvec,
                'tvec': tvec
            })
            
            successful_poses += 1
            print(f"✓ {Path(img_path).name}: Pose estimated (Tag ID: {ids[0][0]})")
        else:
            print(f"✗ {Path(img_path).name}: PnP failed")
    else:
        print(f"✗ {Path(img_path).name}: No tag detected")

print(f"\nSuccessfully estimated poses for {successful_poses}/{len(image_paths)} images")

# Save poses for later use
np.savez('camera_poses.npz', 
         poses=[pose['c2w'] for pose in camera_poses],
         image_paths=[pose['image_path'] for pose in camera_poses])
print("Camera poses saved to 'camera_poses.npz'")

In [ ]:
#visualise
# Start Viser server
server = viser.ViserServer(share=True)
print(f"\nViser server started!")
print(f"Open the provided URL to view the visualization")

# Add a coordinate frame at the origin (ArUco tag position)
server.scene.add_frame(
    "/world",
    wxyz=np.array([1.0, 0.0, 0.0, 0.0]),
    position=np.array([0.0, 0.0, 0.0]),
    axes_length=0.05,
    axes_radius=0.001
)

# Visualize all camera poses
for pose_data in camera_poses:
    i = pose_data['index']
    c2w = pose_data['c2w']
    img = pose_data['image']
    H = pose_data['height']
    W = pose_data['width']
    
    # Convert image to RGB for visualization
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Add camera frustum
    server.scene.add_camera_frustum(
        f"/cameras/{i:03d}",
        fov=2 * np.arctan2(H / 2, camera_matrix[1, 1]),  # vertical FOV
        aspect=W / H,
        scale=0.02,  # Adjust if frustums are too small/large
        wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz,
        position=c2w[:3, 3],
        image=img_rgb
    )

print(f"\nVisualized {len(camera_poses)} camera frustums")
print("Keep this cell running to maintain the visualization...")
print("Take screenshots from different angles for your deliverables!")

# Keep the server running
try:
    while True:
        time.sleep(0.1)
except KeyboardInterrupt:
    print("\nVisualization stopped")

In [ ]:
# Load camera calibration and poses
calib_data = np.load('camera_calibration.npz')
camera_matrix = calib_data['camera_matrix']
dist_coeffs = calib_data['dist_coeffs']

pose_data = np.load('camera_poses.npz', allow_pickle=True)
saved_poses = pose_data['poses']
saved_image_paths = pose_data['image_paths']

print(f"Loaded {len(saved_poses)} camera poses")

# Read a sample image to get dimensions
sample_img = cv2.imread(str(saved_image_paths[0]))
H, W = sample_img.shape[:2]

# IMPORTANT: Get optimal new camera matrix for undistortion
# alpha=0: returns undistorted image with minimum unwanted pixels (crops image)
# alpha=1: returns all pixels, but may have black borders
# alpha=0.5-0.8: good balance (recommended)
new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(
    camera_matrix, 
    dist_coeffs, 
    (W, H), 
    alpha=0.6,  # Adjust this between 0-1
    newImgSize=(W, H)
)

print(f"\nOriginal Camera Matrix:\n{camera_matrix}")
print(f"\nOptimal New Camera Matrix:\n{new_camera_matrix}")
print(f"ROI (region of interest): {roi}")

# Extract focal length from the NEW camera matrix
fx = new_camera_matrix[0, 0]
fy = new_camera_matrix[1, 1]
focal = (fx + fy) / 2.0
print(f"\nFocal length (from new matrix): fx={fx:.2f}, fy={fy:.2f}, average={focal:.2f}")

# Step 1: Undistort all images with the new camera matrix
print("\nUndistorting images...")
undistorted_images = []
c2w_matrices = []

for i, (img_path, c2w) in enumerate(zip(saved_image_paths, saved_poses)):
    # Read original image
    img = cv2.imread(str(img_path))
    
    if img is None:
        print(f"Warning: Could not read {img_path}")
        continue
    
    # Undistort the image using the optimal new camera matrix
    undistorted_img = cv2.undistort(img, camera_matrix, dist_coeffs, None, new_camera_matrix)
    
    # Optional: Crop to ROI to remove black borders
    x, y, w, h = roi
    if w > 0 and h > 0:  # If valid ROI
        undistorted_img = undistorted_img[y:y+h, x:x+w]
    
    # Convert BGR to RGB for NeRF
    undistorted_img_rgb = cv2.cvtColor(undistorted_img, cv2.COLOR_BGR2RGB)
    
    undistorted_images.append(undistorted_img_rgb)
    c2w_matrices.append(c2w)
    
    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/{len(saved_poses)} images")

print(f"Successfully undistorted {len(undistorted_images)} images")

# Convert to numpy arrays
undistorted_images = np.array(undistorted_images, dtype=np.uint8)
c2w_matrices = np.array(c2w_matrices, dtype=np.float32)

print(f"\nDataset shape: {undistorted_images.shape}")
print(f"Poses shape: {c2w_matrices.shape}")

In [ ]:
# Determine split ratios
N_total = len(undistorted_images)
N_train = int(0.8 * N_total)  # 80% for training
N_val = int(0.1 * N_total)    # 10% for validation
N_test = N_total - N_train - N_val  # 10% for testing

print(f"\nDataset split:")
print(f"  Total images: {N_total}")
print(f"  Training: {N_train}")
print(f"  Validation: {N_val}")
print(f"  Test: {N_test}")

# Shuffle indices for random split (optional but recommended)
np.random.seed(42)  # For reproducibility
indices = np.random.permutation(N_total)

# Split indices
train_indices = indices[:N_train]
val_indices = indices[N_train:N_train + N_val]
test_indices = indices[N_train + N_val:]

# Split images and poses
images_train = undistorted_images[train_indices]
c2ws_train = c2w_matrices[train_indices]

images_val = undistorted_images[val_indices]
c2ws_val = c2w_matrices[val_indices]

# For test set, we only need poses (for novel view synthesis)
c2ws_test = c2w_matrices[test_indices]

print(f"\nFinal dataset:")
print(f"  images_train: {images_train.shape}")
print(f"  c2ws_train: {c2ws_train.shape}")
print(f"  images_val: {images_val.shape}")
print(f"  c2ws_val: {c2ws_val.shape}")
print(f"  c2ws_test: {c2ws_test.shape}")
print(f"  focal: {focal}")

In [ ]:
# Save the NeRF dataset
output_filename = 'my_nerf_data.npz'

np.savez(
    output_filename,
    images_train=images_train,    # (N_train, H, W, 3) uint8, 0-255
    c2ws_train=c2ws_train,        # (N_train, 4, 4) float32
    images_val=images_val,        # (N_val, H, W, 3) uint8, 0-255
    c2ws_val=c2ws_val,            # (N_val, 4, 4) float32
    c2ws_test=c2ws_test,          # (N_test, 4, 4) float32
    focal=focal                   # float
)

print(f"\n✓ Dataset saved to '{output_filename}'")
print(f"  File size: {Path(output_filename).stat().st_size / (1024**2):.2f} MB")

In [ ]:
# Load and verify the saved dataset
print("\nVerifying saved dataset...")
loaded_data = np.load(output_filename)

print(f"Keys in dataset: {list(loaded_data.keys())}")
print(f"\nShapes:")
for key in loaded_data.keys():
    data = loaded_data[key]
    if hasattr(data, 'shape'):
        print(f"  {key}: {data.shape}, dtype: {data.dtype}")
    else:
        print(f"  {key}: {data} (scalar)")

# Verify image range
print(f"\nImage value ranges:")
print(f"  Train images: [{loaded_data['images_train'].min()}, {loaded_data['images_train'].max()}]")
print(f"  Val images: [{loaded_data['images_val'].min()}, {loaded_data['images_val'].max()}]")

# Verify poses are valid 4x4 matrices
print(f"\nSample c2w matrix (train[0]):")
print(loaded_data['c2ws_train'][0])

print("\n✓ Dataset verification complete!")

In [ ]:
import matplotlib.pyplot as plt

# Compare original vs undistorted image
original_img = cv2.imread(str(saved_image_paths[20]))
original_img_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)

undistorted_img = cv2.undistort(original_img, camera_matrix, dist_coeffs)
undistorted_img_rgb = cv2.cvtColor(undistorted_img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(original_img_rgb)
axes[0].set_title('Original Image (Distorted)')
axes[0].axis('off')

axes[1].imshow(undistorted_img_rgb)
axes[1].set_title('Undistorted Image')
axes[1].axis('off')

plt.tight_layout()
plt.show()


Part 1:

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load image
img_path = "nerf/IMG_4077.jpg"
img = Image.open(img_path)
img = img.resize((341, 230))
img_array = np.array(img) / 255.0
H, W = img_array.shape[:2]

# Create coordinates
x = torch.linspace(-1, 1, W)
y = torch.linspace(-1, 1, H)
y_grid, x_grid = torch.meshgrid(y, x, indexing='ij')
coords_flat = torch.stack([x_grid, y_grid], dim=-1).reshape(-1, 2).to(device)
img_flat = torch.FloatTensor(img_array.reshape(-1, 3)).to(device)

# Assignment specs
BATCH_SIZE = 10000  # As specified
TARGET_ITERATIONS = 2000  # Choose between 1000-3000
LEARNING_RATE = 1e-2  # As specified

# Calculate number of epochs needed
total_pixels = H * W
iterations_per_epoch = np.ceil(total_pixels / BATCH_SIZE)
num_epochs = int(np.ceil(TARGET_ITERATIONS / iterations_per_epoch))

print(f"Total pixels: {total_pixels}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Iterations per epoch: {iterations_per_epoch}")
print(f"Target iterations: {TARGET_ITERATIONS}")
print(f"Number of epochs needed: {num_epochs}")
print(f"Actual total iterations: {num_epochs * iterations_per_epoch}")

# Create dataloader
dataset = TensorDataset(coords_flat, img_flat)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Model (without PE for now)
class NeuralField(nn.Module):
    def __init__(self, hidden_dim=256, num_layers=4):
        super().__init__()
        layers = []
        layers.append(nn.Linear(2, hidden_dim))
        layers.append(nn.ReLU())
        
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
        
        layers.append(nn.Linear(hidden_dim, 3))
        layers.append(nn.Sigmoid())
        self.network = nn.Sequential(*layers)
    
    def forward(self, coords):
        return self.network(coords)

model = NeuralField(hidden_dim=256, num_layers=4).to(device)

# Optimizer with specified learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

# Training loop (counting iterations, not epochs)
iteration = 0
losses = []
psnrs = []

print("\nStarting training...")
model.train()

for epoch in range(num_epochs):
    for batch_coords, batch_colors in dataloader:
        # Forward pass
        pred_colors = model(batch_coords)
        loss = criterion(pred_colors, batch_colors)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        iteration += 1
        losses.append(loss.item())
        
        # Log progress every 100 iterations
        if iteration % 100 == 0:
            model.eval()
            with torch.no_grad():
                pred_all = model(coords_flat)
                mse = criterion(pred_all, img_flat).item()
                psnr = -10 * np.log10(mse)
                psnrs.append(psnr)
                print(f"Iteration {iteration}/{TARGET_ITERATIONS}: Loss={loss.item():.6f}, PSNR={psnr:.2f} dB")
            model.train()
        
        # Stop if we've reached target iterations
        if iteration >= TARGET_ITERATIONS:
            break
    
    if iteration >= TARGET_ITERATIONS:
        break

print(f"\nTraining complete! Total iterations: {iteration}")

# Final evaluation
model.eval()
with torch.no_grad():
    pred_colors = model(coords_flat)
    pred_img = pred_colors.cpu().numpy().reshape(H, W, 3)
    mse = criterion(pred_colors, img_flat).item()
    final_psnr = -10 * np.log10(mse)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_array)
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(np.clip(pred_img, 0, 1))
axes[1].set_title(f'Reconstruction\n(PSNR: {final_psnr:.2f} dB)')
axes[1].axis('off')

axes[2].plot(losses)
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Loss')
axes[2].set_title('Training Loss')
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
class NeuralFieldPE(nn.Module):
    def __init__(self, num_freqs=6, hidden_dim=256, num_layers=4):
        super().__init__()
        self.num_freqs = num_freqs
        freq_bands = 2.0 ** torch.linspace(0, num_freqs - 1, num_freqs)
        self.register_buffer('freq_bands', freq_bands)
        
        # Input: 2 + 2*2*L (original coords + sin/cos for each freq for x and y)
        input_dim = 2 + 2 * 2 * num_freqs
        
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
        
        layers.append(nn.Linear(hidden_dim, 3))
        layers.append(nn.Sigmoid())
        self.network = nn.Sequential(*layers)
    
    def forward(self, coords):
        # Positional encoding
        encoded = [coords]
        for freq in self.freq_bands:
            encoded.append(torch.sin(2 * np.pi * freq * coords))
            encoded.append(torch.cos(2 * np.pi * freq * coords))
        encoded = torch.cat(encoded, dim=-1)
        return self.network(encoded)

# Train with PE
model_pe = NeuralFieldPE(num_freqs=10, hidden_dim=256, num_layers=4).to(device)
optimizer_pe = torch.optim.Adam(model_pe.parameters(), lr=1e-2)

# Use same training loop as above with model_pe and optimizer_pe

In [ ]:
iteration = 0
losses = []
psnrs = []

print("\nStarting training...")
model_pe.train()

for epoch in range(num_epochs):
    for batch_coords, batch_colors in dataloader:
        # Forward pass
        pred_colors = model_pe(batch_coords)
        loss = criterion(pred_colors, batch_colors)
        
        # Backward pass
        optimizer_pe.zero_grad()
        loss.backward()
        optimizer_pe.step()
        
        iteration += 1
        losses.append(loss.item())
        
        # Log progress every 100 iterations
        if iteration % 100 == 0:
            model_pe.eval()
            with torch.no_grad():
                pred_all = model_pe(coords_flat)
                mse = criterion(pred_all, img_flat).item()
                psnr = -10 * np.log10(mse)
                psnrs.append(psnr)
                print(f"Iteration {iteration}/{TARGET_ITERATIONS}: Loss={loss.item():.6f}, PSNR={psnr:.2f} dB")
            model_pe.train()
        
        # Stop if we've reached target iterations
        if iteration >= TARGET_ITERATIONS:
            break
    
    if iteration >= TARGET_ITERATIONS:
        break

print(f"\nTraining complete! Total iterations: {iteration}")

# Final evaluation
model_pe.eval()
with torch.no_grad():
    pred_colors = model_pe(coords_flat)
    pred_img = pred_colors.cpu().numpy().reshape(H, W, 3)
    mse = criterion(pred_colors, img_flat).item()
    final_psnr = -10 * np.log10(mse)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_array)
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(np.clip(pred_img, 0, 1))
axes[1].set_title(f'Reconstruction\n(PSNR: {final_psnr:.2f} dB)')
axes[1].axis('off')

axes[2].plot(losses)
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Loss')
axes[2].set_title('Training Loss')
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Training function with snapshot capability
def train_with_snapshots(model, dataloader, coords_flat, img_flat, H, W, 
                         num_iterations=2000, snapshot_iterations=[100, 500, 1000, 1500, 2000]):
    """Train model and capture snapshots at specific iterations"""
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    iteration = 0
    snapshots = {}
    losses = []
    psnrs = []
    
    print(f"Training for {num_iterations} iterations...")
    model.train()
    
    # Calculate how many epochs we need
    iterations_per_epoch = len(dataloader)
    num_epochs = int(np.ceil(num_iterations / iterations_per_epoch))
    
    for epoch in range(num_epochs):
        for batch_coords, batch_colors in dataloader:
            # Forward pass
            pred_colors = model(batch_coords)
            loss = criterion(pred_colors, batch_colors)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            iteration += 1
            losses.append(loss.item())
            
            # Save snapshot at specified iterations
            if iteration in snapshot_iterations:
                model.eval()
                with torch.no_grad():
                    pred_all = model(coords_flat)
                    pred_img = pred_all.cpu().numpy().reshape(H, W, 3)
                    mse = criterion(pred_all, img_flat).item()
                    psnr = -10 * np.log10(mse)
                    
                    snapshots[iteration] = {
                        'image': np.clip(pred_img, 0, 1),
                        'psnr': psnr,
                        'loss': loss.item()
                    }
                    print(f"Iteration {iteration}/{num_iterations}: PSNR = {psnr:.2f} dB, Loss = {loss.item():.6f}")
                model.train()
            
            # Stop if reached target iterations
            if iteration >= num_iterations:
                break
        
        if iteration >= num_iterations:
            break
    
    return snapshots, losses, psnrs

# Initialize a FRESH model for training with snapshots
# Use the same architecture as your NeuralFieldPE
model_with_snapshots = NeuralFieldPE(
    num_freqs=6,        # Match your L value
    hidden_dim=256,      # Match your width
    num_layers=4         # Match your number of layers
).to(device)

# Train and capture snapshots
snapshot_iterations = [100, 200, 500, 1000, 2000]

snapshots, losses, psnrs = train_with_snapshots(
    model_with_snapshots,
    dataloader,
    coords_flat,
    img_flat,
    H, W,
    num_iterations=2000,
    snapshot_iterations=snapshot_iterations
)

print("\nTraining complete!")

In [ ]:
# Visualize training progression
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Show original image in top-left
axes[0, 0].imshow(img_array)
axes[0, 0].set_title('Original Image', fontsize=14, fontweight='bold')
axes[0, 0].axis('off')

# Show snapshots
snapshot_list = sorted(snapshots.keys())
for idx, iteration in enumerate(snapshot_list):
    row = (idx + 1) // 3
    col = (idx + 1) % 3
    
    snapshot = snapshots[iteration]
    axes[row, col].imshow(snapshot['image'])
    axes[row, col].set_title(
        f'Iteration {iteration}\nPSNR: {snapshot["psnr"]:.2f} dB', 
        fontsize=12
    )
    axes[row, col].axis('off')

plt.suptitle('Training Progression on fox.jpg', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('training_progression_fox.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Define hyperparameters to test
# Use VERY LOW values to see the effect
freq_values = [2, 10]      # Very low (2) vs High (10)
width_values = [64, 256]   # Very low (64) vs High (256)

print("="*70)
print("Training 2x2 Hyperparameter Grid")
print("="*70)

# Store results
grid_results = {}

for num_freqs in freq_values:
    for hidden_dim in width_values:
        key = f"freq{num_freqs}_width{hidden_dim}"
        print(f"\n[Training] L={num_freqs}, Width={hidden_dim}")
        
        # Create fresh model with these hyperparameters
        model = NeuralFieldPE(
            num_freqs=num_freqs,
            hidden_dim=hidden_dim,
            num_layers=4
        ).to(device)
        
        # Train the model
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()
        
        num_iterations = 2000
        iterations_per_epoch = len(dataloader)
        num_epochs = int(np.ceil(num_iterations / iterations_per_epoch))
        
        iteration = 0
        model.train()
        
        for epoch in range(num_epochs):
            for batch_coords, batch_colors in dataloader:
                pred_colors = model(batch_coords)
                loss = criterion(pred_colors, batch_colors)
                
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
                iteration += 1
                if iteration >= num_iterations:
                    break
            
            if iteration >= num_iterations:
                break
        
        # Evaluate final result
        model.eval()
        with torch.no_grad():
            pred_all = model(coords_flat)
            pred_img = pred_all.cpu().numpy().reshape(H, W, 3)
            mse = criterion(pred_all, img_flat).item()
            psnr = -10 * np.log10(mse)
        
        grid_results[key] = {
            'image': np.clip(pred_img, 0, 1),
            'psnr': psnr,
            'num_freqs': num_freqs,
            'hidden_dim': hidden_dim,
            'num_params': sum(p.numel() for p in model.parameters())
        }
        
        print(f"  ✓ PSNR: {psnr:.2f} dB, Parameters: {grid_results[key]['num_params']:,}")

print("\n" + "="*70)
print("Grid training complete!")
print("="*70)

In [ ]:
# Visualize 2×2 Grid
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

for i, num_freqs in enumerate(freq_values):
    for j, hidden_dim in enumerate(width_values):
        key = f"freq{num_freqs}_width{hidden_dim}"
        result = grid_results[key]
        
        axes[i, j].imshow(result['image'])
        axes[i, j].set_title(
            f'L={num_freqs}, Width={hidden_dim}\n'
            f'PSNR: {result["psnr"]:.2f} dB\n'
            f'Params: {result["num_params"]:,}',
            fontsize=12,
            fontweight='bold'
        )
        axes[i, j].axis('off')

plt.suptitle('Hyperparameter Comparison: Frequency (L) vs Width', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('hyperparameter_grid_2x2.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary table
print("\nHyperparameter Comparison Summary:")
print("-" * 70)
print(f"{'L (Freq)':<10} {'Width':<10} {'PSNR (dB)':<12} {'Parameters':<15}")
print("-" * 70)
for key in sorted(grid_results.keys()):
    result = grid_results[key]
    print(f"{result['num_freqs']:<10} {result['hidden_dim']:<10} "
          f"{result['psnr']:<12.2f} {result['num_params']:<15,}")
print("-" * 70)

In [ ]:
# Show comparison with original image
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Original image (top-left)
axes[0, 0].imshow(img_array)
axes[0, 0].axis('off')

# Show each configuration
positions = [(0, 1), (0, 2), (1, 0), (1, 1)]
for idx, (key, result) in enumerate(sorted(grid_results.items())):
    row, col = positions[idx]
    axes[row, col].imshow(result['image'])
    axes[row, col].set_title(
        f'L={result["num_freqs"]}, W={result["hidden_dim"]}\n'
        f'PSNR: {result["psnr"]:.2f} dB',
        fontsize=11
    )
    axes[row, col].axis('off')

# Hide the last subplot
axes[1, 2].axis('off')

plt.suptitle('Comparison: Original vs Different Hyperparameters', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('hyperparameter_comparison_with_original.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Train one model and track PSNR throughout training
print("="*70)
print("Training with PSNR Tracking")
print("="*70)

# Choose your configuration (use the best one or any you prefer)
num_freqs_for_curve = 10
hidden_dim_for_curve = 256

print(f"Training model with L={num_freqs_for_curve}, Width={hidden_dim_for_curve}")

# Create model
model_for_psnr = NeuralFieldPE(
    num_freqs=num_freqs_for_curve,
    hidden_dim=hidden_dim_for_curve,
    num_layers=4
).to(device)

optimizer = torch.optim.Adam(model_for_psnr.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Training setup
num_iterations = 2000
iterations_per_epoch = len(dataloader)
num_epochs = int(np.ceil(num_iterations / iterations_per_epoch))

# Track metrics
iteration = 0
losses = []
psnrs = []
psnr_iterations = []

model_for_psnr.train()

for epoch in range(num_epochs):
    for batch_coords, batch_colors in dataloader:
        # Forward pass
        pred_colors = model_for_psnr(batch_coords)
        loss = criterion(pred_colors, batch_colors)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_for_psnr.parameters(), max_norm=1.0)
        optimizer.step()
        
        iteration += 1
        losses.append(loss.item())
        
        # Compute PSNR every 10 iterations
        if iteration % 10 == 0:
            model_for_psnr.eval()
            with torch.no_grad():
                pred_all = model_for_psnr(coords_flat)
                mse = criterion(pred_all, img_flat).item()
                psnr = -10 * np.log10(mse)
                psnrs.append(psnr)
                psnr_iterations.append(iteration)
            model_for_psnr.train()
        
        # Print progress
        if iteration % 200 == 0:
            print(f"  Iteration {iteration}/{num_iterations}: PSNR = {psnrs[-1]:.2f} dB")
        
        if iteration >= num_iterations:
            break
    
    if iteration >= num_iterations:
        break

print(f"\nTraining complete! Final PSNR: {psnrs[-1]:.2f} dB")

In [ ]:
# Plot PSNR curve
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# PSNR curve
axes[0].plot(psnr_iterations, psnrs, linewidth=2, color='#2E86AB', marker='o', 
             markersize=3, markevery=20)
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('PSNR (dB)', fontsize=12)
axes[0].set_title(f'PSNR Training Curve\n(L={num_freqs_for_curve}, Width={hidden_dim_for_curve})', 
                  fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, num_iterations)

# Add final PSNR annotation
axes[0].annotate(f'Final: {psnrs[-1]:.2f} dB',
                xy=(psnr_iterations[-1], psnrs[-1]),
                xytext=(-60, -20), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

# Loss curve
axes[1].plot(losses, linewidth=1.5, color='#A23B72', alpha=0.8)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('MSE Loss', fontsize=12)
axes[1].set_title('Training Loss Curve', fontsize=14, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, num_iterations)

plt.tight_layout()
plt.savefig('psnr_and_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Optional: Show PSNR improvement over time with visual examples
# Sample at different points in training
sample_iterations = [200, 600, 1000, 1400, 2000]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Original
axes[0, 0].imshow(img_array)
axes[0, 0].set_title('Original Image', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Find closest PSNR values to our sample points
for idx, target_iter in enumerate(sample_iterations):
    row = (idx + 1) // 3
    col = (idx + 1) % 3
    
    # Find closest iteration in our PSNR tracking
    closest_idx = min(range(len(psnr_iterations)), 
                     key=lambda i: abs(psnr_iterations[i] - target_iter))
    actual_iter = psnr_iterations[closest_idx]
    psnr_value = psnrs[closest_idx]
    
    axes[row, col].plot(psnr_iterations[:closest_idx+1], psnrs[:closest_idx+1], 
                       linewidth=2, color='#2E86AB')
    axes[row, col].scatter([actual_iter], [psnr_value], color='red', s=100, zorder=5)
    axes[row, col].set_xlabel('Iteration', fontsize=10)
    axes[row, col].set_ylabel('PSNR (dB)', fontsize=10)
    axes[row, col].set_title(f'Iteration {actual_iter}\nPSNR: {psnr_value:.2f} dB', 
                            fontsize=11)
    axes[row, col].grid(True, alpha=0.3)
    axes[row, col].set_xlim(0, num_iterations)

plt.suptitle('PSNR Improvement Over Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('psnr_progression.png', dpi=150, bbox_inches='tight')
plt.show()

Part 2:

In [ ]:
# parsing data from lego dataset
data = np.load(f"lego_200x200.npz")

# Training images: [100, 200, 200, 3]
images_train = data["images_train"] / 255.0

# Cameras for the training images 
# (camera-to-world transformation matrix): [100, 4, 4]
c2ws_train = data["c2ws_train"]

# Validation images: 
images_val = data["images_val"] / 255.0

# Cameras for the validation images: [10, 4, 4]
# (camera-to-world transformation matrix): [10, 200, 200, 3]
c2ws_val = data["c2ws_val"]

# Test cameras for novel-view video rendering: 
# (camera-to-world transformation matrix): [60, 4, 4]
c2ws_test = data["c2ws_test"]

# Camera focal length
focal = data["focal"]  # float

In [ ]:

def transform(c2w, x_c):
    """
    Transform points from camera space to world space (or apply any 4x4 transformation).
    
    Args:
        c2w: Camera-to-world transformation matrix
             Shape: (4, 4) or (B, 4, 4) for batched transformations
        x_c: Points in camera space
             Shape: (3,) or (N, 3) or (B, N, 3) for batched points
             
    Returns:
        x_w: Points in world space
             Same shape as x_c
    """
    # Convert to torch if numpy
    if isinstance(c2w, np.ndarray):
        c2w = torch.from_numpy(c2w).float()
    if isinstance(x_c, np.ndarray):
        x_c = torch.from_numpy(x_c).float()
    
    # Store original shape to restore later
    original_shape = x_c.shape
    
    # Handle different input shapes
    # Case 1: Single point (3,)
    if x_c.dim() == 1:
        x_c = x_c.unsqueeze(0)  # (1, 3)
        squeeze_output = True
    else:
        squeeze_output = False
    
    # Case 2: Single transformation matrix with multiple points
    # c2w: (4, 4), x_c: (N, 3)
    if c2w.dim() == 2 and x_c.dim() == 2:
        # Convert to homogeneous coordinates: (N, 3) -> (N, 4)
        ones = torch.ones(x_c.shape[0], 1, device=x_c.device, dtype=x_c.dtype)
        x_c_homogeneous = torch.cat([x_c, ones], dim=-1)  # (N, 4)
        
        # Apply transformation: (4, 4) @ (N, 4).T -> (4, N) -> (N, 4)
        x_w_homogeneous = (c2w @ x_c_homogeneous.T).T  # (N, 4)
        
        # Convert back from homogeneous: (N, 4) -> (N, 3)
        x_w = x_w_homogeneous[..., :3]
    
    # Case 3: Batched transformations with batched points
    # c2w: (B, 4, 4), x_c: (B, N, 3)
    elif c2w.dim() == 3 and x_c.dim() == 3:
        # Convert to homogeneous coordinates: (B, N, 3) -> (B, N, 4)
        ones = torch.ones(*x_c.shape[:-1], 1, device=x_c.device, dtype=x_c.dtype)
        x_c_homogeneous = torch.cat([x_c, ones], dim=-1)  # (B, N, 4)
        
        # Apply transformation: (B, 4, 4) @ (B, N, 4).T -> (B, 4, N) -> (B, N, 4)
        x_w_homogeneous = torch.bmm(c2w, x_c_homogeneous.transpose(-2, -1)).transpose(-2, -1)
        
        # Convert back from homogeneous: (B, N, 4) -> (B, N, 3)
        x_w = x_w_homogeneous[..., :3]
    
    else:
        raise ValueError(f"Incompatible shapes: c2w {c2w.shape}, x_c {x_c.shape}")
    
    # Restore original shape if needed
    if squeeze_output:
        x_w = x_w.squeeze(0)
    
    return x_w

In [ ]:
# Test 1: Single point transformation
print("Test 1: Single point transformation")
print("="*60)

# Create a simple camera-to-world matrix (identity + translation)
c2w = torch.eye(4)
c2w[:3, 3] = torch.tensor([1.0, 2.0, 3.0])  # Translation

# Test point in camera space
x_c = torch.tensor([1.0, 0.0, 0.0])

# Transform to world space
x_w = transform(c2w, x_c)
print(f"Point in camera space: {x_c}")
print(f"Point in world space: {x_w}")
print(f"Expected: [2.0, 2.0, 3.0]")
print()

# Test 2: Inverse transformation verification
print("Test 2: Inverse transformation verification")
print("="*60)

# Random camera-to-world matrix
c2w = torch.randn(4, 4)
c2w[3, :] = torch.tensor([0, 0, 0, 1])  # Ensure last row is [0, 0, 0, 1]

# Random point
x = torch.randn(3)

# Transform to world and back to camera
x_w = transform(c2w, x)
c2w_inv = torch.linalg.inv(c2w)
x_reconstructed = transform(c2w_inv, x_w)

# Check if they're equal
difference = torch.abs(x - x_reconstructed).max()
print(f"Original point: {x}")
print(f"After c2w -> inv(c2w): {x_reconstructed}")
print(f"Max difference: {difference:.10f}")
print(f"Are they equal? {torch.allclose(x, x_reconstructed, atol=1e-6)}")
print()

# Test 3: Batch of points with single transformation
print("Test 3: Batch of points (single camera)")
print("="*60)

c2w = torch.eye(4)
c2w[:3, 3] = torch.tensor([1.0, 2.0, 3.0])

# Multiple points
x_c_batch = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
])

x_w_batch = transform(c2w, x_c_batch)
print(f"Points in camera space:\n{x_c_batch}")
print(f"Points in world space:\n{x_w_batch}")
print()

# Test 4: Batched transformations with batched points
print("Test 4: Batched transformations")
print("="*60)

# Batch of 2 cameras
c2w_batch = torch.eye(4).unsqueeze(0).repeat(2, 1, 1)
c2w_batch[0, :3, 3] = torch.tensor([1.0, 0.0, 0.0])
c2w_batch[1, :3, 3] = torch.tensor([0.0, 1.0, 0.0])

# Batch of 2 sets of points (each set has 3 points)
x_c_batch = torch.ones(2, 3, 3)

x_w_batch = transform(c2w_batch, x_c_batch)
print(f"Batched c2w shape: {c2w_batch.shape}")
print(f"Batched points shape: {x_c_batch.shape}")
print(f"Transformed points shape: {x_w_batch.shape}")
print(f"Transformed points:\n{x_w_batch}")
print()

# Test 5: Using actual dataset
print("Test 5: Using actual dataset")
print("="*60)

# Load your data
data = np.load("lego_200x200.npz")
c2ws_train = torch.from_numpy(data["c2ws_train"]).float()

# Test with first camera
c2w_first = c2ws_train[0]
print(f"First camera c2w matrix:\n{c2w_first}")

# Create test points
test_points = torch.tensor([
    [0.0, 0.0, 0.0],  # Origin
    [1.0, 0.0, 0.0],  # Unit x
    [0.0, 1.0, 0.0],  # Unit y
    [0.0, 0.0, 1.0],  # Unit z
])

world_points = transform(c2w_first, test_points)
print(f"\nTest points in camera space:\n{test_points}")
print(f"Test points in world space:\n{world_points}")

# Verify inverse
c2w_first_inv = torch.linalg.inv(c2w_first)
reconstructed = transform(c2w_first_inv, world_points)
print(f"\nReconstructed points (should match original):\n{reconstructed}")
print(f"Max reconstruction error: {torch.abs(test_points - reconstructed).max():.10f}")
print()

In [ ]:

def pixel_to_camera(K, uv, s):
    """
    Transform points from pixel coordinate system to camera coordinate system.
    
    The forward projection is: s * [u, v, 1]^T = K @ [x_c, y_c, z_c]^T
    We need to invert this to get: [x_c, y_c, z_c]^T = (1/s) * K^(-1) @ [u, v, 1]^T
    
    Args:
        K: Camera intrinsic matrix (3, 3)
        uv: Pixel coordinates (2,) or (N, 2) or (B, N, 2)
            [u, v] where u is column (x-axis) and v is row (y-axis)
        s: Depth values (scalar, (N,), or (B, N))
            Depth along the optical axis (z_c in camera coordinates)
    
    Returns:
        x_c: Points in camera coordinate system (3,) or (N, 3) or (B, N, 3)
             [x_c, y_c, z_c]
    """
    # Convert to torch if numpy
    if isinstance(K, np.ndarray):
        K = torch.from_numpy(K).float()
    if isinstance(uv, np.ndarray):
        uv = torch.from_numpy(uv).float()
    if isinstance(s, np.ndarray):
        s = torch.from_numpy(s).float()
    
    # Store original shape info
    original_uv_shape = uv.shape
    
    # Handle scalar depth
    if isinstance(s, (int, float)):
        s = torch.tensor(s, dtype=torch.float32)
    
    # Ensure s has compatible shape
    if s.dim() == 0:  # Scalar
        s = s.unsqueeze(0)
        s_was_scalar = True
    else:
        s_was_scalar = False
    
    # Handle different input shapes for uv
    squeeze_output = False
    if uv.dim() == 1:  # Single point (2,)
        uv = uv.unsqueeze(0)  # (1, 2)
        squeeze_output = True
        if not s_was_scalar and s.shape[0] > 1:
            s = s[:1]  # Take first depth value
    
    # At this point:
    # uv should be (N, 2) or (B, N, 2)
    # s should be (N,) or (B, N) or (1,) for scalar case
    
    # Compute inverse of K
    K_inv = torch.linalg.inv(K)  # (3, 3)
    
    # Case 1: Single batch - uv: (N, 2), s: (N,) or scalar
    if uv.dim() == 2:
        N = uv.shape[0]
        
        # Convert to homogeneous coordinates: (N, 2) -> (N, 3)
        ones = torch.ones(N, 1, device=uv.device, dtype=uv.dtype)
        uv_homogeneous = torch.cat([uv, ones], dim=-1)  # (N, 3)
        
        # Apply inverse intrinsic matrix: (3, 3) @ (N, 3).T -> (3, N) -> (N, 3)
        x_c_normalized = (K_inv @ uv_homogeneous.T).T  # (N, 3)
        
        # Scale by depth
        # Ensure s has shape (N, 1) for broadcasting
        if s.dim() == 1:
            s = s.unsqueeze(-1)  # (N, 1)
        
        x_c = x_c_normalized * s  # (N, 3)
    
    # Case 2: Batched - uv: (B, N, 2), s: (B, N)
    elif uv.dim() == 3:
        B, N = uv.shape[0], uv.shape[1]
        
        # Convert to homogeneous coordinates: (B, N, 2) -> (B, N, 3)
        ones = torch.ones(B, N, 1, device=uv.device, dtype=uv.dtype)
        uv_homogeneous = torch.cat([uv, ones], dim=-1)  # (B, N, 3)
        
        # Apply inverse intrinsic matrix
        # K_inv: (3, 3), uv_homogeneous: (B, N, 3)
        # We need to do: K_inv @ uv_homogeneous for each batch
        x_c_normalized = torch.einsum('ij,bnj->bni', K_inv, uv_homogeneous)  # (B, N, 3)
        
        # Scale by depth
        # Ensure s has shape (B, N, 1) for broadcasting
        if s.dim() == 2:
            s = s.unsqueeze(-1)  # (B, N, 1)
        
        x_c = x_c_normalized * s  # (B, N, 3)
    
    else:
        raise ValueError(f"Incompatible uv shape: {uv.shape}")
    
    # Restore original output shape if needed
    if squeeze_output:
        x_c = x_c.squeeze(0)  # (3,)
    
    return x_c

In [ ]:
# Test 1: Simple case with identity-like K matrix
print("Test 1: Simple verification")
print("="*60)

# Create a simple intrinsic matrix
focal = 100.0
W, H = 200, 200
o_x, o_y = W / 2, H / 2

K = torch.tensor([
    [focal, 0, o_x],
    [0, focal, o_y],
    [0, 0, 1]
], dtype=torch.float32)

print(f"Intrinsic matrix K:\n{K}\n")

# Test point in camera space
x_c_original = torch.tensor([1.0, 2.0, 5.0])  # [x, y, z]

# Forward: camera to pixel
# s * [u, v, 1]^T = K @ [x, y, z]^T
x_c_homogeneous = x_c_original
s = x_c_original[2]  # depth is z_c
uv_homogeneous = K @ x_c_homogeneous
uv = uv_homogeneous[:2] / uv_homogeneous[2]  # Normalize by depth

print(f"Original point in camera space: {x_c_original}")
print(f"Depth (s): {s}")
print(f"Projected to pixel space: {uv}")

# Backward: pixel to camera
x_c_reconstructed = pixel_to_camera(K, uv, s)

print(f"Reconstructed point in camera space: {x_c_reconstructed}")
print(f"Difference: {torch.abs(x_c_original - x_c_reconstructed).max():.10f}")
print(f"Are they equal? {torch.allclose(x_c_original, x_c_reconstructed, atol=1e-6)}")
print()

# Test 2: Batch of points
print("Test 2: Batch of points")
print("="*60)

# Multiple points in camera space
x_c_batch = torch.tensor([
    [1.0, 0.0, 5.0],
    [0.0, 1.0, 5.0],
    [-1.0, -1.0, 10.0],
    [2.0, 3.0, 8.0]
])

# Forward projection for each point
depths = x_c_batch[:, 2]  # z coordinates
uv_batch = []

for i in range(x_c_batch.shape[0]):
    x_c_hom = x_c_batch[i]
    uv_hom = K @ x_c_hom
    uv_batch.append(uv_hom[:2] / uv_hom[2])

uv_batch = torch.stack(uv_batch)

print(f"Original points in camera space:\n{x_c_batch}")
print(f"\nProjected to pixel space:\n{uv_batch}")
print(f"\nDepths: {depths}")

# Backward projection
x_c_reconstructed_batch = pixel_to_camera(K, uv_batch, depths)

print(f"\nReconstructed points:\n{x_c_reconstructed_batch}")
print(f"Max difference: {torch.abs(x_c_batch - x_c_reconstructed_batch).max():.10f}")
print()

# Test 3: Using actual dataset focal length
print("Test 3: Using actual dataset")
print("="*60)

# Load your data
data = np.load("lego_200x200.npz")
focal_actual = float(data["focal"])
W, H = 200, 200

# Create K matrix from dataset
K_actual = torch.tensor([
    [focal_actual, 0, W/2],
    [0, focal_actual, H/2],
    [0, 0, 1]
], dtype=torch.float32)

print(f"Actual focal length: {focal_actual}")
print(f"K matrix:\n{K_actual}\n")

# Test some pixel coordinates
test_pixels = torch.tensor([
    [100.0, 100.0],  # Center pixel
    [0.0, 0.0],      # Top-left corner
    [199.0, 199.0],  # Bottom-right corner
    [150.0, 75.0]    # Arbitrary pixel
])

test_depths = torch.tensor([1.0, 1.0, 1.0, 2.0])

x_c_from_pixels = pixel_to_camera(K_actual, test_pixels, test_depths)

print(f"Pixel coordinates:\n{test_pixels}")
print(f"Depths: {test_depths}")
print(f"Camera coordinates:\n{x_c_from_pixels}\n")

# Verify by projecting back to pixels
for i in range(test_pixels.shape[0]):
    x_c = x_c_from_pixels[i]
    uv_hom = K_actual @ x_c
    uv_reconstructed = uv_hom[:2] / uv_hom[2]
    print(f"Pixel {i}: {test_pixels[i]} -> Camera: {x_c} -> Pixel: {uv_reconstructed}")
    print(f"  Difference: {torch.abs(test_pixels[i] - uv_reconstructed).max():.10f}")

print()

# Test 4: All pixels in an image
print("Test 4: All pixels in an image")
print("="*60)

# Create pixel grid
u = torch.arange(W, dtype=torch.float32)
v = torch.arange(H, dtype=torch.float32)
v_grid, u_grid = torch.meshgrid(v, u, indexing='ij')

# Flatten to (H*W, 2)
uv_all = torch.stack([u_grid.flatten(), v_grid.flatten()], dim=-1)  # (40000, 2)

# Use constant depth for all pixels
depth_constant = 5.0

# Convert all pixels to camera coordinates
x_c_all = pixel_to_camera(K_actual, uv_all, depth_constant)

print(f"Total pixels: {uv_all.shape[0]}")
print(f"Camera coordinates shape: {x_c_all.shape}")
print(f"Sample camera coordinates (first 5):\n{x_c_all[:5]}")
print(f"All z coordinates should be {depth_constant}: {torch.allclose(x_c_all[:, 2], torch.tensor(depth_constant))}")
print()

# Test 5: Verify the mathematical relationship
print("Test 5: Mathematical verification")
print("="*60)

# For a point at pixel (u, v) with depth s:
# The camera coordinate should satisfy: K @ x_c = s * [u, v, 1]^T

u, v = 120.5, 80.3
s_test = 3.5

uv_test = torch.tensor([u, v])
x_c_test = pixel_to_camera(K_actual, uv_test, s_test)

# Verify: K @ x_c should equal s * [u, v, 1]
lhs = K_actual @ x_c_test
rhs = s_test * torch.tensor([u, v, 1.0])

print(f"Pixel: ({u}, {v}), Depth: {s_test}")
print(f"Camera coordinate: {x_c_test}")
print(f"K @ x_c = {lhs}")
print(f"s * [u, v, 1] = {rhs}")
print(f"Are they equal? {torch.allclose(lhs, rhs, atol=1e-5)}")

In [ ]:
def pixel_to_ray(K, c2w, uv):
    """
    Convert pixel coordinates to rays with origin and normalized direction.
    
    For a pinhole camera:
    - Ray origin (r_o) is the camera center in world coordinates: r_o = t (translation from c2w)
    - Ray direction (r_d) is computed by:
      1. Convert pixel to camera coordinates at depth s=1: X_c = pixel_to_camera(K, uv, s=1)
      2. Transform to world coordinates: X_w = transform(c2w, X_c)
      3. Compute direction: r_d = (X_w - r_o) / ||X_w - r_o||
    
    Args:
        K: Camera intrinsic matrix (3, 3)
        c2w: Camera-to-world transformation matrix (4, 4) or (B, 4, 4)
        uv: Pixel coordinates (2,) or (N, 2) or (B, N, 2)
            [u, v] where u is column and v is row
    
    Returns:
        ray_o: Ray origins in world coordinates (3,) or (N, 3) or (B, N, 3)
        ray_d: Ray directions (normalized) in world coordinates (3,) or (N, 3) or (B, N, 3)
    """
    # Convert to torch if numpy
    if isinstance(K, np.ndarray):
        K = torch.from_numpy(K).float()
    if isinstance(c2w, np.ndarray):
        c2w = torch.from_numpy(c2w).float()
    if isinstance(uv, np.ndarray):
        uv = torch.from_numpy(uv).float()
    
    # Handle different input shapes
    squeeze_output = False
    if uv.dim() == 1:  # Single pixel (2,)
        uv = uv.unsqueeze(0)  # (1, 2)
        squeeze_output = True
    
    # Case 1: Single camera with multiple pixels
    # c2w: (4, 4), uv: (N, 2)
    if c2w.dim() == 2 and uv.dim() == 2:
        N = uv.shape[0]
        
        # Step 1: Get ray origin from camera translation
        # r_o = t (the translation component of c2w)
        ray_o = c2w[:3, 3]  # (3,)
        ray_o = ray_o.unsqueeze(0).expand(N, 3)  # (N, 3) - same origin for all rays
        
        # Step 2: Convert pixel to camera coordinates at depth s=1
        s = torch.ones(N, device=uv.device, dtype=uv.dtype)
        x_c = pixel_to_camera(K, uv, s)  # (N, 3)
        
        # Step 3: Transform camera coordinates to world coordinates
        x_w = transform(c2w, x_c)  # (N, 3)
        
        # Step 4: Compute ray direction and normalize
        ray_d = x_w - ray_o  # (N, 3)
        ray_d = ray_d / torch.norm(ray_d, dim=-1, keepdim=True)  # Normalize (N, 3)
    
    # Case 2: Batched cameras with batched pixels
    # c2w: (B, 4, 4), uv: (B, N, 2)
    elif c2w.dim() == 3 and uv.dim() == 3:
        B, N = uv.shape[0], uv.shape[1]
        
        # Step 1: Get ray origins from camera translations
        # r_o = t for each camera
        ray_o = c2w[:, :3, 3]  # (B, 3)
        ray_o = ray_o.unsqueeze(1).expand(B, N, 3)  # (B, N, 3) - broadcast to all pixels
        
        # Step 2: Convert pixels to camera coordinates at depth s=1
        s = torch.ones(B, N, device=uv.device, dtype=uv.dtype)
        
        # We need to apply pixel_to_camera for each batch
        # Create batched K if needed
        K_batched = K.unsqueeze(0).expand(B, 3, 3)  # (B, 3, 3)
        
        # Convert each batch
        x_c_list = []
        for b in range(B):
            x_c_b = pixel_to_camera(K, uv[b], s[b])  # (N, 3)
            x_c_list.append(x_c_b)
        x_c = torch.stack(x_c_list, dim=0)  # (B, N, 3)
        
        # Step 3: Transform to world coordinates
        x_w = transform(c2w, x_c)  # (B, N, 3)
        
        # Step 4: Compute ray direction and normalize
        ray_d = x_w - ray_o  # (B, N, 3)
        ray_d = ray_d / torch.norm(ray_d, dim=-1, keepdim=True)  # Normalize (B, N, 3)
    
    # Case 3: Single camera (4, 4) with single pixel after unsqueeze
    elif c2w.dim() == 2 and uv.shape[0] == 1:
        # This is handled by Case 1
        ray_o = c2w[:3, 3].unsqueeze(0)  # (1, 3)
        
        s = torch.ones(1, device=uv.device, dtype=uv.dtype)
        x_c = pixel_to_camera(K, uv, s)  # (1, 3)
        x_w = transform(c2w, x_c)  # (1, 3)
        
        ray_d = x_w - ray_o  # (1, 3)
        ray_d = ray_d / torch.norm(ray_d, dim=-1, keepdim=True)
    
    else:
        raise ValueError(f"Incompatible shapes: c2w {c2w.shape}, uv {uv.shape}")
    
    # Restore original shape if needed
    if squeeze_output:
        ray_o = ray_o.squeeze(0)  # (3,)
        ray_d = ray_d.squeeze(0)  # (3,)
    
    return ray_o, ray_d

In [ ]:
# Test 1: Single pixel, single camera
print("Test 1: Single pixel ray")
print("="*60)

# Load dataset
data = np.load("lego_200x200.npz")
focal = float(data["focal"])
c2ws_train = torch.from_numpy(data["c2ws_train"]).float()

W, H = 200, 200
K = torch.tensor([
    [focal, 0, W/2],
    [0, focal, H/2],
    [0, 0, 1]
], dtype=torch.float32)

# Use first camera
c2w = c2ws_train[0]

# Test center pixel
uv_center = torch.tensor([W/2, H/2])

ray_o, ray_d = pixel_to_ray(K, c2w, uv_center)

print(f"Camera c2w matrix:\n{c2w}\n")
print(f"Pixel coordinates (center): {uv_center}")
print(f"Ray origin: {ray_o}")
print(f"Ray direction: {ray_d}")
print(f"Ray direction norm (should be 1.0): {torch.norm(ray_d):.6f}")
print(f"Camera center from c2w: {c2w[:3, 3]}")
print(f"Ray origin matches camera center: {torch.allclose(ray_o, c2w[:3, 3])}")
print()

# Test 2: Multiple pixels, single camera
print("Test 2: Multiple pixels")
print("="*60)

# Test corner and center pixels
test_pixels = torch.tensor([
    [0.0, 0.0],        # Top-left corner
    [W-1, 0.0],        # Top-right corner
    [0.0, H-1],        # Bottom-left corner
    [W-1, H-1],        # Bottom-right corner
    [W/2, H/2],        # Center
])

ray_o_batch, ray_d_batch = pixel_to_ray(K, c2w, test_pixels)

print(f"Test pixels:\n{test_pixels}\n")
print(f"Ray origins shape: {ray_o_batch.shape}")
print(f"Ray directions shape: {ray_d_batch.shape}\n")

# Verify all origins are the same (same camera)
print("All ray origins should be identical (same camera):")
for i in range(ray_o_batch.shape[0]):
    print(f"  Ray {i}: origin = {ray_o_batch[i]}")
print()

print("Ray directions (should be different and normalized):")
for i in range(ray_d_batch.shape[0]):
    norm = torch.norm(ray_d_batch[i])
    print(f"  Ray {i}: direction = {ray_d_batch[i]}, norm = {norm:.6f}")
print()

# Test 3: All pixels in an image
print("Test 3: All pixels in image")
print("="*60)

# Create pixel grid
u = torch.arange(W, dtype=torch.float32)
v = torch.arange(H, dtype=torch.float32)
v_grid, u_grid = torch.meshgrid(v, u, indexing='ij')
uv_all = torch.stack([u_grid.flatten(), v_grid.flatten()], dim=-1)  # (H*W, 2)

print(f"Total pixels: {uv_all.shape[0]} ({H}x{W})")

ray_o_all, ray_d_all = pixel_to_ray(K, c2w, uv_all)

print(f"Ray origins shape: {ray_o_all.shape}")
print(f"Ray directions shape: {ray_d_all.shape}")
print(f"All ray directions normalized: {torch.allclose(torch.norm(ray_d_all, dim=-1), torch.ones(ray_d_all.shape[0]))}")
print(f"All ray origins identical: {torch.allclose(ray_o_all, ray_o_all[0].unsqueeze(0).expand_as(ray_o_all))}")
print()

# Test 4: Visualize ray directions
print("Test 4: Ray direction properties")
print("="*60)

# Center pixel should point roughly along camera's forward direction (z-axis in camera space)
# Camera's forward direction in world space is the 3rd column of rotation matrix (but negated in some conventions)
camera_forward = c2w[:3, 2]  # or -c2w[:3, 2] depending on convention
camera_right = c2w[:3, 0]
camera_up = c2w[:3, 1]

print(f"Camera forward direction: {camera_forward}")
print(f"Camera right direction: {camera_right}")
print(f"Camera up direction: {camera_up}")

# Get ray for center pixel
uv_center = torch.tensor([W/2, H/2])
_, ray_d_center = pixel_to_ray(K, c2w, uv_center)

print(f"\nCenter pixel ray direction: {ray_d_center}")
print(f"Dot product with camera forward: {torch.dot(ray_d_center, camera_forward):.4f}")
print(f"(Should be close to 1.0 or -1.0 depending on convention)")
print()

# Test 5: Verify ray passes through correct world point
print("Test 5: Verify ray geometry")
print("="*60)

# Pick a pixel
uv_test = torch.tensor([120.0, 80.0])

# Get ray for this pixel
ray_o_test, ray_d_test = pixel_to_ray(K, c2w, uv_test)

print(f"Test pixel: {uv_test}")
print(f"Ray origin: {ray_o_test}")
print(f"Ray direction: {ray_d_test}")

# Any point along this ray can be parameterized as: P(t) = ray_o + t * ray_d
# Let's verify that if we choose a specific depth, convert pixel to camera to world,
# that point should lie on the ray

depth = 3.5
x_c = pixel_to_camera(K, uv_test, depth)
x_w = transform(c2w, x_c)

print(f"\nPoint at depth {depth}:")
print(f"  Camera coordinates: {x_c}")
print(f"  World coordinates: {x_w}")

# This point should lie on the ray: x_w = ray_o + t * ray_d
# Solve for t: t = ||x_w - ray_o|| (since ray_d is normalized)
t = torch.norm(x_w - ray_o_test)
point_on_ray = ray_o_test + t * ray_d_test

print(f"\nPoint on ray at t={t:.4f}: {point_on_ray}")
print(f"Should match world point: {x_w}")
print(f"Difference: {torch.norm(x_w - point_on_ray):.10f}")
print(f"Points match: {torch.allclose(x_w, point_on_ray, atol=1e-5)}")
print()

# Test 6: Batched cameras (if you want to support this)
print("Test 6: Multiple cameras")
print("="*60)

# Use first 3 cameras
c2ws_batch = c2ws_train[:3]  # (3, 4, 4)

# Same pixel for each camera
uv_single = torch.tensor([W/2, H/2])
uv_repeated = uv_single.unsqueeze(0).expand(3, 1, 2)  # (3, 1, 2)

# Get rays for each camera
ray_o_multi, ray_d_multi = pixel_to_ray(K, c2ws_batch, uv_repeated)

print(f"Testing center pixel for 3 different cameras")
print(f"Ray origins shape: {ray_o_multi.shape}")
print(f"Ray directions shape: {ray_d_multi.shape}\n")

print("Ray origins (should be different - different camera positions):")
for i in range(3):
    print(f"  Camera {i}: {ray_o_multi[i, 0]}")

print("\nRay directions (should be different - different camera orientations):")
for i in range(3):
    print(f"  Camera {i}: {ray_d_multi[i, 0]}, norm = {torch.norm(ray_d_multi[i, 0]):.6f}")

In [ ]:
def to_torch(x):
    if isinstance(x, np.ndarray):
        return torch.from_numpy(x).float()
    elif isinstance(x, torch.Tensor):
        return x.float()
    else:
        raise TypeError(f"Expected np.ndarray or torch.Tensor, got {type(x)}")


In [ ]:
class RaysData:
    """
    Dataset for NeRF that precomputes all rays from training images.
    This stores all rays in memory for fast sampling.
    """
    def __init__(self, images, K, c2ws):
        """
        Args:
            images: (N, H, W, 3) numpy array of training images in [0, 1]
            K: (3, 3) numpy array or torch tensor of camera intrinsic matrix
            c2ws: (N, 4, 4) numpy array of camera-to-world matrices
        """
        self.images = images
        
        # Convert K to numpy if it's a tensor
        if isinstance(K, torch.Tensor):
            K = K.cpu().numpy()
        self.K = K
        
        self.c2ws = c2ws
        
        N, H, W, _ = images.shape
        self.N = N
        self.H = H
        self.W = W
        
        print(f"Initializing RaysData with {N} images of size {H}x{W}")
        
        # Create pixel coordinate grid with 0.5 offset for pixel centers
        # Note: u is width (x), v is height (y)
        u = np.arange(W) + 0.5  # Add 0.5 for pixel centers
        v = np.arange(H) + 0.5
        u_grid, v_grid = np.meshgrid(u, v)  # u_grid and v_grid are both (H, W)
        
        # Store UVs as integer coordinates (without the 0.5 offset) for indexing
        # These are used to index into the image array
        u_int = np.arange(W)
        v_int = np.arange(H)
        u_grid_int, v_grid_int = np.meshgrid(u_int, v_int)
        
        # Flatten and create (H*W, 2) array of [u, v] integer coordinates for ALL pixels in one image
        self.uvs = np.stack([u_grid_int.flatten(), v_grid_int.flatten()], axis=-1)  # (H*W, 2)
        
        # Now precompute all rays for all images
        print("Precomputing all rays...")
        all_rays_o = []
        all_rays_d = []
        all_pixels = []
        
        for img_idx in range(N):
            # Get camera matrix for this image
            c2w = c2ws[img_idx]
            
            # Create UV coordinates with 0.5 offset for ray generation
            uv_with_offset = np.stack([u_grid.flatten(), v_grid.flatten()], axis=-1)  # (H*W, 2)
            
            # Convert to torch for computation
            c2w_torch = to_torch(c2w)
            K_torch = to_torch(self.K)
            uv_torch = torch.from_numpy(uv_with_offset).float()
            
            # Generate rays for all pixels in this image
            ray_o, ray_d = pixel_to_ray(K_torch, c2w_torch, uv_torch)
            
            # Convert back to numpy
            ray_o_np = ray_o.cpu().numpy()  # (H*W, 3)
            ray_d_np = ray_d.cpu().numpy()  # (H*W, 3)
            
            # Get pixel colors
            pixels = images[img_idx].reshape(-1, 3)  # (H*W, 3)
            
            all_rays_o.append(ray_o_np)
            all_rays_d.append(ray_d_np)
            all_pixels.append(pixels)
            
            if (img_idx + 1) % 10 == 0:
                print(f"  Processed {img_idx + 1}/{N} images")
        
        # Stack all rays: (N*H*W, 3)
        self.rays_o = np.concatenate(all_rays_o, axis=0)
        self.rays_d = np.concatenate(all_rays_d, axis=0)
        self.pixels = np.concatenate(all_pixels, axis=0)
        
        # Repeat uvs for all images
        self.uvs = np.tile(self.uvs, (N, 1))  # (N*H*W, 2)
        
        print(f"Precomputed {self.rays_o.shape[0]} rays")
        print(f"  rays_o shape: {self.rays_o.shape}")
        print(f"  rays_d shape: {self.rays_d.shape}")
        print(f"  pixels shape: {self.pixels.shape}")
        print(f"  uvs shape: {self.uvs.shape}")
    
    def sample_rays(self, num_rays):
        """
        Randomly sample rays from all precomputed rays.
        
        Args:
            num_rays: Number of rays to sample
            
        Returns:
            rays_o: (num_rays, 3) ray origins
            rays_d: (num_rays, 3) ray directions
            pixels: (num_rays, 3) pixel colors
        """
        # Randomly sample indices
        total_rays = self.rays_o.shape[0]
        indices = np.random.randint(0, total_rays, size=num_rays)
        
        rays_o = self.rays_o[indices]
        rays_d = self.rays_d[indices]
        pixels = self.pixels[indices]
        
        return rays_o, rays_d, pixels
    
    def get_rays_from_image(self, img_idx):
        """
        Get all rays from a specific image.
        
        Args:
            img_idx: Image index
            
        Returns:
            rays_o: (H*W, 3)
            rays_d: (H*W, 3)
            pixels: (H*W, 3)
        """
        start_idx = img_idx * self.H * self.W
        end_idx = start_idx + self.H * self.W
        
        return self.rays_o[start_idx:end_idx], self.rays_d[start_idx:end_idx], self.pixels[start_idx:end_idx]

In [ ]:
def sample_along_rays(rays_o, rays_d, near=2.0, far=6.0, n_samples=64, random=True):
    """
    Sample points along rays.
    
    Args:
        rays_o: (N, 3) ray origins
        rays_d: (N, 3) ray directions
        near: Near bound (default: 2.0)
        far: Far bound (default: 6.0)
        n_samples: Number of samples per ray (default: 64)
        random: Whether to use stratified sampling with randomness (True for training)
        
    Returns:
        points: (N, n_samples, 3) sampled points in 3D
    """
    N = rays_o.shape[0]
    
    # Convert to torch if numpy
    if isinstance(rays_o, np.ndarray):
        rays_o = torch.from_numpy(rays_o).float()
    if isinstance(rays_d, np.ndarray):
        rays_d = torch.from_numpy(rays_d).float()
    
    # Create bin edges for stratified sampling
    t_vals = torch.linspace(near, far, n_samples + 1)
    
    if random:
        # Stratified sampling: sample uniformly within each bin
        lower = t_vals[:-1].unsqueeze(0).expand(N, n_samples)  # (N, n_samples)
        upper = t_vals[1:].unsqueeze(0).expand(N, n_samples)   # (N, n_samples)
        
        # Random uniform samples in each bin
        rand = torch.rand(N, n_samples)
        t_vals = lower + (upper - lower) * rand  # (N, n_samples)
    else:
        # Use bin centers (for validation)
        t_vals = (t_vals[:-1] + t_vals[1:]) / 2.0
        t_vals = t_vals.unsqueeze(0).expand(N, n_samples)
    
    # Compute 3D points: P = ray_o + t * ray_d
    # rays_o: (N, 3) -> (N, 1, 3)
    # rays_d: (N, 3) -> (N, 1, 3)
    # t_vals: (N, n_samples) -> (N, n_samples, 1)
    points = rays_o.unsqueeze(1) + t_vals.unsqueeze(-1) * rays_d.unsqueeze(1)
    # points: (N, n_samples, 3)
    
    # Convert back to numpy
    return points.numpy()

In [ ]:
def to_numpy(x):
    """Convert torch tensor or keep numpy array."""
    if isinstance(x, torch.Tensor):
        return x.cpu().numpy()
    return x

In [ ]:
# Sample rays
dataset = RaysData(images_train, K, c2ws_train)
rays_o, rays_d, pixels = dataset.sample_rays(100)
points = sample_along_rays(rays_o, rays_d, random=True)

H, W = images_train.shape[1:3]
K_np = to_numpy(K)

server = viser.ViserServer(share=True)
print("Viser server started! Open the URL above.")

# Add cameras
for i, (image, c2w) in enumerate(zip(images_train, c2ws_train)):
    server.scene.add_camera_frustum(
        f"/cameras/{i}",
        fov=float(2 * np.arctan2(H / 2, K_np[0, 0])),
        aspect=float(W / H),
        scale=0.15,
        wxyz=viser.transforms.SO3.from_matrix(to_numpy(c2w)[:3, :3]).wxyz,
        position=to_numpy(c2w)[:3, 3],
        image=to_numpy(image)
    )

# Add rays
for i, (o, d) in enumerate(zip(rays_o, rays_d)):
    positions = np.stack((to_numpy(o), to_numpy(o) + to_numpy(d) * 6.0))
    server.scene.add_spline_catmull_rom(f"/rays/{i}", positions=positions)

# Add points
server.scene.add_point_cloud(
    f"/samples",
    colors=np.zeros_like(to_numpy(points)).reshape(-1, 3),
    points=to_numpy(points).reshape(-1, 3),
    point_size=0.02,
)

print("Visualization ready!")
print("Press Ctrl+C to stop...")

try:
    while True:
        time.sleep(0.1)
except KeyboardInterrupt:
    print("\nVisualization stopped")

In [ ]:
# many rays from one camera
# Setup
dataset = RaysData(images_train, K, c2ws_train)
CAMERA_INDEX = 0  # ← Change this to see different cameras
NUM_RAYS = 300    # ← Change this to see more/fewer rays

H, W = images_train.shape[1:3]
K_np = to_numpy(K)

server = viser.ViserServer(share=True)

# Add cameras
for i, (image, c2w) in enumerate(zip(images_train, c2ws_train)):
    server.scene.add_camera_frustum(
        f"/cameras/{i}",
        fov=float(2 * np.arctan2(H / 2, K_np[0, 0])),
        aspect=float(W / H),
        scale=0.15,
        wxyz=viser.transforms.SO3.from_matrix(to_numpy(c2w)[:3, :3]).wxyz,
        position=to_numpy(c2w)[:3, 3],
        image=to_numpy(image)
    )

# Get rays from ONE camera
rays_o_all, rays_d_all, _ = dataset.get_rays_from_image(CAMERA_INDEX)
indices = np.random.randint(0, H*W, size=NUM_RAYS)
rays_o = rays_o_all[indices]
rays_d = rays_d_all[indices]

# Sample points
points = sample_along_rays(rays_o, rays_d, random=True)

# Visualize rays
for i, (o, d) in enumerate(zip(rays_o, rays_d)):
    server.scene.add_spline_catmull_rom(
        f"/rays/{i}",
        positions=np.stack((o, o + d * 6.0)),
    )

# Visualize points
server.scene.add_point_cloud(
    "/samples",
    colors=np.zeros_like(points).reshape(-1, 3),
    points=points.reshape(-1, 3),
    point_size=0.02,
)

print(f"Showing {NUM_RAYS} rays from camera {CAMERA_INDEX}")
print("Press Ctrl+C to stop...")

try:
    while True:
        time.sleep(0.1)
except KeyboardInterrupt:
    pass

In [ ]:

class PositionalEncoding(nn.Module):
    """
    Positional encoding for input coordinates.
    """
    def __init__(self, num_freqs):
        """
        Args:
            num_freqs: Number of frequency bands (L)
        """
        super().__init__()
        self.num_freqs = num_freqs
        
        # Create frequency bands: [2^0, 2^1, ..., 2^(L-1)]
        freq_bands = 2.0 ** torch.linspace(0, num_freqs - 1, num_freqs)
        self.register_buffer('freq_bands', freq_bands)
    
    def forward(self, x):
        """
        Apply positional encoding.
        
        Args:
            x: Input tensor of shape (..., D) where D is the input dimension
            
        Returns:
            Encoded tensor of shape (..., D + D*2*num_freqs)
        """
        # x: (..., D)
        encoded = [x]  # Start with original input
        
        # Add sin and cos for each frequency
        for freq in self.freq_bands:
            encoded.append(torch.sin(2 * np.pi * freq * x))
            encoded.append(torch.cos(2 * np.pi * freq * x))
        
        # Concatenate along last dimension
        return torch.cat(encoded, dim=-1)
    
    def get_output_dim(self, input_dim):
        """Calculate output dimension after encoding."""
        return input_dim + input_dim * 2 * self.num_freqs


class NeRFMLP(nn.Module):
    """
    NeRF MLP network that predicts density and RGB color from 3D coordinates and view direction.
    """
    def __init__(self, pos_enc_freqs=10, dir_enc_freqs=4, hidden_dim=256, num_layers=8):
        """
        Args:
            pos_enc_freqs: Number of frequencies for position encoding (L=10)
            dir_enc_freqs: Number of frequencies for direction encoding (L=4)
            hidden_dim: Hidden layer dimension (default: 256)
            num_layers: Total number of layers (default: 8)
        """
        super().__init__()
        
        self.pos_enc_freqs = pos_enc_freqs
        self.dir_enc_freqs = dir_enc_freqs
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Positional encoding for 3D coordinates
        self.pos_encoder = PositionalEncoding(pos_enc_freqs)
        pos_input_dim = self.pos_encoder.get_output_dim(3)  # 3 + 3*2*L
        
        # Positional encoding for 3D ray directions
        self.dir_encoder = PositionalEncoding(dir_enc_freqs)
        dir_input_dim = self.dir_encoder.get_output_dim(3)  # 3 + 3*2*L
        
        print(f"Position encoding output dim: {pos_input_dim}")
        print(f"Direction encoding output dim: {dir_input_dim}")
        
        # First part of the network (before skip connection)
        # Input: encoded position
        self.layers_before_skip = nn.ModuleList()
        
        # First layer
        self.layers_before_skip.append(nn.Linear(pos_input_dim, hidden_dim))
        self.layers_before_skip.append(nn.ReLU())
        
        # Layers 2-4 (3 more layers before skip connection at layer 5)
        for i in range(3):
            self.layers_before_skip.append(nn.Linear(hidden_dim, hidden_dim))
            self.layers_before_skip.append(nn.ReLU())
        
        # Skip connection layer (layer 5)
        # Concatenate: hidden features + encoded position
        self.skip_layer = nn.Linear(hidden_dim + pos_input_dim, hidden_dim)
        self.skip_relu = nn.ReLU()
        
        # Second part of the network (after skip connection)
        # Layers 6-8 (3 more layers)
        self.layers_after_skip = nn.ModuleList()
        for i in range(3):
            self.layers_after_skip.append(nn.Linear(hidden_dim, hidden_dim))
            self.layers_after_skip.append(nn.ReLU())
        
        # Density prediction head
        # Output: 1D density (sigma)
        self.density_head = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.ReLU()  # Density must be positive
        )
        
        # Feature layer before color prediction
        # This layer prepares features for color prediction
        self.feature_layer = nn.Linear(hidden_dim, hidden_dim)
        
        # Color prediction head
        # Input: features + encoded direction
        # Output: 3D RGB color
        self.color_head = nn.Sequential(
            nn.Linear(hidden_dim + dir_input_dim, hidden_dim // 2),  # 256 + dir_dim -> 128
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 3),  # 128 -> 3
            nn.Sigmoid()  # RGB values in [0, 1]
        )
    
    def forward(self, x, ray_d):
        """
        Forward pass of NeRF MLP.
        
        Args:
            x: 3D coordinates, shape (..., 3)
            ray_d: Ray directions, shape (..., 3) - should be normalized
            
        Returns:
            density: Density values, shape (..., 1)
            rgb: RGB colors, shape (..., 3)
        """
        # Encode inputs
        x_encoded = self.pos_encoder(x)  # (..., pos_input_dim)
        ray_d_encoded = self.dir_encoder(ray_d)  # (..., dir_input_dim)
        
        # First part: layers before skip connection
        h = x_encoded
        for layer in self.layers_before_skip:
            h = layer(h)
        
        # Skip connection: concatenate with original encoded input
        h = torch.cat([h, x_encoded], dim=-1)  # (..., hidden_dim + pos_input_dim)
        h = self.skip_layer(h)
        h = self.skip_relu(h)
        
        # Second part: layers after skip connection
        for layer in self.layers_after_skip:
            h = layer(h)
        
        # Predict density (does not depend on view direction)
        density = self.density_head(h)  # (..., 1)
        
        # Prepare features for color prediction
        h_color = self.feature_layer(h)  # (..., hidden_dim)
        
        # Concatenate with encoded direction for color prediction
        h_color = torch.cat([h_color, ray_d_encoded], dim=-1)
        
        # Predict RGB color (depends on view direction)
        rgb = self.color_head(h_color)  # (..., 3)
        
        return density, rgb


# Test the network
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("="*70)
    print("Testing NeRF MLP")
    print("="*70)
    
    # Create model
    model = NeRFMLP(
        pos_enc_freqs=10,  # L=10 for positions
        dir_enc_freqs=4,   # L=4 for directions
        hidden_dim=256,
        num_layers=8
    ).to(device)
    
    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    print(f"\nTotal parameters: {num_params:,}")
    
    # Test with random inputs
    batch_size = 1024
    num_samples = 64
    
    # Create test data
    # Positions: (batch_size, num_samples, 3)
    positions = torch.randn(batch_size, num_samples, 3).to(device)
    
    # Ray directions: (batch_size, 3) - same direction for all samples along a ray
    ray_dirs = torch.randn(batch_size, 3).to(device)
    ray_dirs = ray_dirs / torch.norm(ray_dirs, dim=-1, keepdim=True)  # Normalize
    
    # Expand ray directions to match positions shape
    ray_dirs_expanded = ray_dirs.unsqueeze(1).expand(-1, num_samples, -1)  # (batch_size, num_samples, 3)
    
    print(f"\nTest input shapes:")
    print(f"  Positions: {positions.shape}")
    print(f"  Ray directions: {ray_dirs_expanded.shape}")
    
    # Forward pass
    with torch.no_grad():
        density, rgb = model(positions, ray_dirs_expanded)
    
    print(f"\nOutput shapes:")
    print(f"  Density: {density.shape}")
    print(f"  RGB: {rgb.shape}")
    
    print(f"\nOutput ranges:")
    print(f"  Density: [{density.min():.4f}, {density.max():.4f}] (should be >= 0)")
    print(f"  RGB: [{rgb.min():.4f}, {rgb.max():.4f}] (should be in [0, 1])")
    
    # Verify density is non-negative
    assert (density >= 0).all(), "Density contains negative values!"
    
    # Verify RGB is in [0, 1]
    assert (rgb >= 0).all() and (rgb <= 1).all(), "RGB not in [0, 1] range!"
    
    print("\n✓ All tests passed!")

In [ ]:

def volrend(sigmas, rgbs, step_size):
    """
    Volume rendering using the NeRF rendering equation.
    
    Args:
        sigmas: Density values, shape (N_rays, N_samples, 1)
        rgbs: RGB colors, shape (N_rays, N_samples, 3)
        step_size: Distance between samples (scalar)
        
    Returns:
        rendered_colors: RGB colors for each ray, shape (N_rays, 3)
    """
    # sigmas: (N_rays, N_samples, 1)
    # rgbs: (N_rays, N_samples, 3)
    
    # Compute alpha values: alpha_i = 1 - exp(-sigma_i * delta_i)
    # where delta_i is the step size
    alphas = 1.0 - torch.exp(-sigmas * step_size)  # (N_rays, N_samples, 1)
    
    # Compute transmittance T_i = exp(-sum_{j=1}^{i-1} sigma_j * delta_j)
    # This is the probability of a ray NOT terminating before sample i
    
    # First, compute sigma * delta for each sample
    sigma_delta = sigmas * step_size  # (N_rays, N_samples, 1)
    
    # Cumulative sum to get sum_{j=1}^{i-1} sigma_j * delta_j
    # We need to shift by one position (no accumulation for first sample)
    cumsum_sigma_delta = torch.cumsum(sigma_delta, dim=1)  # (N_rays, N_samples, 1)
    
    # Shift: T_i uses cumsum up to i-1, so we prepend zeros and remove last element
    cumsum_sigma_delta = torch.cat([
        torch.zeros_like(cumsum_sigma_delta[:, :1, :]),  # (N_rays, 1, 1) of zeros
        cumsum_sigma_delta[:, :-1, :]  # Remove last element
    ], dim=1)  # (N_rays, N_samples, 1)
    
    # Compute transmittance
    transmittance = torch.exp(-cumsum_sigma_delta)  # (N_rays, N_samples, 1)
    
    # Compute weights: w_i = T_i * (1 - exp(-sigma_i * delta_i))
    weights = transmittance * alphas  # (N_rays, N_samples, 1)
    
    # Compute rendered color: C = sum_{i=1}^{N} w_i * c_i
    rendered_colors = torch.sum(weights * rgbs, dim=1)  # (N_rays, 3)
    
    return rendered_colors


# Test the volume rendering function
print("Testing volume rendering...")
torch.manual_seed(42)
sigmas = torch.rand((10, 64, 1))
rgbs = torch.rand((10, 64, 3))
step_size = (6.0 - 2.0) / 64
rendered_colors = volrend(sigmas, rgbs, step_size)

correct = torch.tensor([
    [0.5006, 0.3728, 0.4728],
    [0.4322, 0.3559, 0.4134],
    [0.4027, 0.4394, 0.4610],
    [0.4514, 0.3829, 0.4196],
    [0.4002, 0.4599, 0.4103],
    [0.4471, 0.4044, 0.4069],
    [0.4285, 0.4072, 0.3777],
    [0.4152, 0.4190, 0.4361],
    [0.4051, 0.3651, 0.3969],
    [0.3253, 0.3587, 0.4215]
])

assert torch.allclose(rendered_colors, correct, rtol=1e-4, atol=1e-4), "Volume rendering test failed!"
print("✓ Volume rendering test passed!")

In [ ]:
import torch.optim as optim
from tqdm import tqdm

def train_nerf(
    images_train, c2ws_train, images_val, c2ws_val, K,
    num_iterations=5000,
    batch_size=10000,  # 10K rays per batch
    n_samples=64,
    learning_rate=5e-4,
    near=2.0,
    far=6.0,
    device='cuda',
    validate_every=100,
    save_visualization_every=200
):
    """
    Train NeRF model.
    
    Args:
        images_train: Training images (N, H, W, 3)
        c2ws_train: Training camera poses (N, 4, 4)
        images_val: Validation images (M, H, W, 3)
        c2ws_val: Validation camera poses (M, 4, 4)
        K: Camera intrinsic matrix (3, 3)
        num_iterations: Number of training iterations
        batch_size: Number of rays per batch
        n_samples: Number of samples per ray
        learning_rate: Learning rate for Adam optimizer
        near: Near plane distance
        far: Far plane distance
        device: 'cuda' or 'cpu'
        validate_every: Validate every N iterations
        save_visualization_every: Save visualization every N iterations
    """
    
    # Create dataset
    print("Creating training dataset...")
    train_dataset = RaysData(images_train, K, c2ws_train)
    
    # Create validation rays (render full validation images)
    print("Creating validation dataset...")
    val_dataset = RaysData(images_val, K, c2ws_val)
    
    H, W = images_train.shape[1:3]
    
    # Initialize model
    print("Initializing NeRF model...")
    model = NeRFMLP(
        pos_enc_freqs=10,
        dir_enc_freqs=4,
        hidden_dim=256,
        num_layers=8
    ).to(device)
    
    print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")
    
    # Optimizer
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Loss function
    criterion = nn.MSELoss()
    
    # Training tracking
    train_losses = []
    val_psnrs = []
    val_iterations = []
    visualizations = []
    
    step_size = (far - near) / n_samples
    
    print(f"\nStarting training for {num_iterations} iterations...")
    print(f"Batch size: {batch_size} rays")
    print(f"Samples per ray: {n_samples}")
    print(f"Learning rate: {learning_rate}")
    print("="*70)
    
    start_time = time.time()
    
    for iteration in tqdm(range(num_iterations)):
        model.train()
        
        # Sample rays
        rays_o, rays_d, rgb_gt = train_dataset.sample_rays(batch_size)
        
        # Convert to torch tensors
        rays_o = torch.from_numpy(rays_o).float().to(device)
        rays_d = torch.from_numpy(rays_d).float().to(device)
        rgb_gt = torch.from_numpy(rgb_gt).float().to(device)
        
        # Sample points along rays
        points = sample_along_rays_torch(rays_o, rays_d, near, far, n_samples, perturb=True, device=device)
        # points: (batch_size, n_samples, 3)
        
        # Expand ray directions to match points shape
        rays_d_expanded = rays_d.unsqueeze(1).expand(-1, n_samples, -1)
        # rays_d_expanded: (batch_size, n_samples, 3)
        
        # Forward pass through NeRF
        density, rgb = model(points, rays_d_expanded)
        # density: (batch_size, n_samples, 1)
        # rgb: (batch_size, n_samples, 3)
        
        # Volume rendering
        rgb_rendered = volrend(density, rgb, step_size)
        # rgb_rendered: (batch_size, 3)
        
        # Compute loss
        loss = criterion(rgb_rendered, rgb_gt)
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
        
        # Validation
        if (iteration + 1) % validate_every == 0:
            model.eval()
            
            # Compute PSNR on validation set
            psnr = evaluate_nerf(model, val_dataset, images_val, K, 
                                n_samples, near, far, device)
            
            val_psnrs.append(psnr)
            val_iterations.append(iteration + 1)
            
            elapsed_time = time.time() - start_time
            print(f"\nIteration {iteration+1}/{num_iterations}")
            print(f"  Train Loss: {loss.item():.6f}")
            print(f"  Val PSNR: {psnr:.2f} dB")
            print(f"  Time: {elapsed_time/60:.1f} min")
        
        # Save visualization
        if (iteration + 1) % save_visualization_every == 0 or iteration == 0:
            model.eval()
            with torch.no_grad():
                # Render first validation image
                val_img_rendered = render_image(
                    model, c2ws_val[0], K, H, W, 
                    n_samples, near, far, device
                )
                visualizations.append({
                    'iteration': iteration + 1,
                    'image': val_img_rendered,
                    'psnr': val_psnrs[-1] if val_psnrs else 0
                })
    
    total_time = time.time() - start_time
    print(f"\nTraining complete! Total time: {total_time/60:.1f} minutes")
    
    return model, train_losses, val_psnrs, val_iterations, visualizations


def sample_along_rays_torch(rays_o, rays_d, near, far, n_samples, perturb=True, device='cuda'):
    """
    Sample points along rays (torch version for training).
    """
    N_rays = rays_o.shape[0]
    
    # Create bin edges
    t_vals = torch.linspace(near, far, n_samples + 1, device=device)
    
    if perturb:
        # Stratified sampling
        lower = t_vals[:-1].unsqueeze(0).expand(N_rays, n_samples)
        upper = t_vals[1:].unsqueeze(0).expand(N_rays, n_samples)
        rand = torch.rand(N_rays, n_samples, device=device)
        t_vals = lower + (upper - lower) * rand
    else:
        # Use bin centers
        t_vals = (t_vals[:-1] + t_vals[1:]) / 2.0
        t_vals = t_vals.unsqueeze(0).expand(N_rays, n_samples)
    
    # Compute points
    points = rays_o.unsqueeze(1) + t_vals.unsqueeze(-1) * rays_d.unsqueeze(1)
    
    return points


def evaluate_nerf(model, val_dataset, images_val, K, n_samples, near, far, device, max_rays=10000):
    """
    Evaluate NeRF on validation set and compute PSNR.
    """
    model.eval()
    
    H, W = images_val.shape[1:3]
    step_size = (far - near) / n_samples
    
    psnrs = []
    
    with torch.no_grad():
        for img_idx in range(len(images_val)):
            # Get all rays for this validation image
            rays_o_all, rays_d_all, rgb_gt_all = val_dataset.get_rays_from_image(img_idx)
            
            # Render in chunks to avoid OOM
            rgb_rendered_chunks = []
            
            for i in range(0, len(rays_o_all), max_rays):
                rays_o = torch.from_numpy(rays_o_all[i:i+max_rays]).float().to(device)
                rays_d = torch.from_numpy(rays_d_all[i:i+max_rays]).float().to(device)
                
                # Sample points
                points = sample_along_rays_torch(rays_o, rays_d, near, far, n_samples, perturb=False, device=device)
                
                # Expand directions
                rays_d_expanded = rays_d.unsqueeze(1).expand(-1, n_samples, -1)
                
                # Forward pass
                density, rgb = model(points, rays_d_expanded)
                
                # Volume rendering
                rgb_rendered = volrend(density, rgb, step_size)
                
                rgb_rendered_chunks.append(rgb_rendered.cpu().numpy())
            
            # Concatenate all chunks
            rgb_rendered_full = np.concatenate(rgb_rendered_chunks, axis=0)
            
            # Compute MSE and PSNR
            mse = np.mean((rgb_rendered_full - rgb_gt_all) ** 2)
            psnr = -10 * np.log10(mse)
            psnrs.append(psnr)
    
    return np.mean(psnrs)


def render_image(model, c2w, K, H, W, n_samples, near, far, device, chunk_size=4096):
    """
    Render a full image from a given camera pose.
    """
    model.eval()
    
    # Convert K to numpy if needed
    K_np = K.cpu().numpy() if isinstance(K, torch.Tensor) else K
    
    # Create pixel grid
    u = np.arange(W) + 0.5
    v = np.arange(H) + 0.5
    u_grid, v_grid = np.meshgrid(u, v)
    uv = np.stack([u_grid.flatten(), v_grid.flatten()], axis=-1)
    
    # Convert to torch
    c2w_torch = torch.from_numpy(c2w).float() if isinstance(c2w, np.ndarray) else c2w
    K_torch = torch.from_numpy(K_np).float()
    uv_torch = torch.from_numpy(uv).float()
    
    # Generate all rays
    rays_o, rays_d = pixel_to_ray(K_torch, c2w_torch, uv_torch)
    rays_o = rays_o.to(device)
    rays_d = rays_d.to(device)
    
    step_size = (far - near) / n_samples
    
    # Render in chunks
    rgb_rendered_chunks = []
    
    with torch.no_grad():
        for i in range(0, len(rays_o), chunk_size):
            rays_o_chunk = rays_o[i:i+chunk_size]
            rays_d_chunk = rays_d[i:i+chunk_size]
            
            # Sample points
            points = sample_along_rays_torch(rays_o_chunk, rays_d_chunk, near, far, n_samples, perturb=False, device=device)
            
            # Expand directions
            rays_d_expanded = rays_d_chunk.unsqueeze(1).expand(-1, n_samples, -1)
            
            # Forward pass
            density, rgb = model(points, rays_d_expanded)
            
            # Volume rendering
            rgb_rendered = volrend(density, rgb, step_size)
            
            rgb_rendered_chunks.append(rgb_rendered.cpu())
    
    # Concatenate and reshape
    rgb_rendered = torch.cat(rgb_rendered_chunks, dim=0)
    rgb_rendered = rgb_rendered.reshape(H, W, 3).numpy()
    
    return np.clip(rgb_rendered, 0, 1)

In [ ]:
def visualize_training_progress(visualizations, images_val, save_path='training_progress.png'):
    """
    Visualize training progress with rendered images at different iterations.
    """
    n_vis = len(visualizations)
    
    fig, axes = plt.subplots(2, (n_vis + 1) // 2, figsize=(4 * ((n_vis + 1) // 2), 8))
    axes = axes.flatten()
    
    for idx, vis in enumerate(visualizations):
        axes[idx].imshow(vis['image'])
        axes[idx].set_title(f"Iter {vis['iteration']}\nPSNR: {vis['psnr']:.2f} dB", fontsize=10)
        axes[idx].axis('off')
    
    # Show ground truth in last subplot
    if n_vis < len(axes):
        axes[-1].imshow(images_val[0])
        axes[-1].set_title('Ground Truth', fontsize=10)
        axes[-1].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_psnr_curve(val_iterations, val_psnrs, save_path='psnr_curve.png'):
    """
    Plot PSNR curve over training.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(val_iterations, val_psnrs, linewidth=2, marker='o', markersize=4)
    plt.xlabel('Iteration', fontsize=12)
    plt.ylabel('PSNR (dB)', fontsize=12)
    plt.title('Validation PSNR over Training', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def render_novel_view_video(model, c2ws_test, K, H, W, n_samples, near, far, device, output_path='novel_views.mp4'):
    """
    Render novel views and create a video.
    """
    import imageio
    
    print(f"Rendering {len(c2ws_test)} novel views...")
    frames = []
    
    for i, c2w in enumerate(tqdm(c2ws_test)):
        img = render_image(model, c2w, K, H, W, n_samples, near, far, device)
        frames.append((img * 255).astype(np.uint8))
    
    # Save video
    imageio.mimsave(output_path, frames, fps=30)
    print(f"Video saved to {output_path}")
    
    return frames

In [ ]:
def to_tensor(x, device):
    if isinstance(x, np.ndarray):
        return torch.from_numpy(x).float().to(device)
    elif isinstance(x, torch.Tensor):
        return x.float().to(device)
    else:
        raise TypeError(f"Expected np.ndarray or torch.Tensor, got {type(x)}")


In [ ]:
def train_nerf_fast(
    images_train, c2ws_train, images_val, c2ws_val, K,
    num_iterations=1000,
    batch_size=4096,
    n_samples=48,
    learning_rate=5e-4,
    near=2.0,
    far=6.0,
    device='cpu'   
):
    """
    Optimized fast training for NeRF with checkpoint saving and snapshot rendering.
    """
    import os, time, torch, numpy as np
    import torch.nn as nn, torch.optim as optim
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    from pathlib import Path

    # Prepare folders
    Path("checkpoints").mkdir(exist_ok=True)
    Path("progress_images").mkdir(exist_ok=True)

    H, W = images_train.shape[1:3]
    print("Creating dataset...")
    train_dataset = RaysData(images_train, K, c2ws_train)

    print("Initializing model...")
    model = NeRFMLP(
        pos_enc_freqs=10,
        dir_enc_freqs=4,
        hidden_dim=256,
        num_layers=8
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    step_size = (far - near) / n_samples

    train_losses, val_psnrs, val_iterations = [], [], []

    start_iter = 0

    print("Pre-sampling validation rays...")
    val_rays_o, val_rays_d, val_rgb_gt = [], [], []
    for img_idx in range(len(images_val)):
        rays_o, rays_d, rgb_gt = RaysData(
            images_val[img_idx:img_idx+1], K, c2ws_val[img_idx:img_idx+1]
        ).get_rays_from_image(0)
        val_rays_o.append(rays_o)
        val_rays_d.append(rays_d)
        val_rgb_gt.append(rgb_gt)

    print("Starting training...")
    start_time = time.time()

    for iteration in tqdm(range(start_iter, num_iterations)):
        model.train()

        # === Sample random training rays ===
        rays_o, rays_d, rgb_gt = train_dataset.sample_rays(batch_size)
        rays_o = to_tensor(rays_o, device)
        rays_d = to_tensor(rays_d, device)
        rgb_gt = to_tensor(rgb_gt, device)

        # === Stratified sampling along each ray ===
        with torch.no_grad():
            t_vals = torch.linspace(near, far, n_samples + 1, device=device)
            lower = t_vals[:-1].unsqueeze(0).expand(batch_size, n_samples)
            upper = t_vals[1:].unsqueeze(0).expand(batch_size, n_samples)
            rand = torch.rand(batch_size, n_samples, device=device)
            t_vals = lower + (upper - lower) * rand
            points = rays_o.unsqueeze(1) + t_vals.unsqueeze(-1) * rays_d.unsqueeze(1)

        # === Forward pass ===
        rays_d_expanded = rays_d.unsqueeze(1).expand(-1, n_samples, -1)
        density, rgb = model(points, rays_d_expanded)
        rgb_rendered = volrend(density, rgb, step_size)

        # === Compute loss ===
        loss = criterion(rgb_rendered, rgb_gt)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

        # === Validation every 100 iterations ===
        if (iteration + 1) % 100 == 0 or iteration + 1 == num_iterations:
            model.eval()
            with torch.no_grad():
                n_val_rays = 1000
                val_idx = np.random.randint(0, len(val_rays_o[0]), n_val_rays)
                rays_o_val = to_tensor(val_rays_o[0][val_idx], device)
                rays_d_val = to_tensor(val_rays_d[0][val_idx], device)
                rgb_gt_val = to_tensor(val_rgb_gt[0][val_idx], device)


                t_vals = torch.linspace(near, far, n_samples + 1, device=device)
                t_vals = (t_vals[:-1] + t_vals[1:]) / 2.0
                t_vals = t_vals.unsqueeze(0).expand(n_val_rays, n_samples)
                points = rays_o_val.unsqueeze(1) + t_vals.unsqueeze(-1) * rays_d_val.unsqueeze(1)
                rays_d_expanded = rays_d_val.unsqueeze(1).expand(-1, n_samples, -1)
                density, rgb = model(points, rays_d_expanded)
                rgb_rendered = volrend(density, rgb, step_size)

                mse = torch.mean((rgb_rendered - rgb_gt_val) ** 2).item()
                psnr = -10 * np.log10(mse)

                val_psnrs.append(psnr)
                val_iterations.append(iteration + 1)
                elapsed = time.time() - start_time
                print(f"\nIter {iteration+1}: Loss={loss.item():.6f}, PSNR={psnr:.2f} dB, Time={elapsed/60:.1f}min")

            # === Save checkpoint ===
            ckpt_path = f"checkpoints/nerf_iter{iteration+1}.pth"
            torch.save({
                "iteration": iteration + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_losses": train_losses,
                "val_psnrs": val_psnrs,
                "val_iterations": val_iterations,
            }, ckpt_path)
            print(f"✅ Saved checkpoint: {ckpt_path}")

            # === Save predicted validation image ===
            # Render full validation image 0 (fewer rays for speed)
            if (iteration + 1) in [1, 100, 200, 500, 1000, 1500, num_iterations]:
                print(f"Rendering progress image at iter {iteration+1}...")
                # use subset of pixels to make faster
                H_s, W_s = H//4, W//4
                xs, ys = torch.meshgrid(
                    torch.linspace(0, W-1, W_s),
                    torch.linspace(0, H-1, H_s),
                    indexing='xy'
                )
                rays_o_img, rays_d_img, _ = RaysData(images_val[:1], K, c2ws_val[:1]).get_rays_from_image(0)
                rays_o_img = to_tensor(rays_o_img, device)
                rays_d_img = to_tensor(rays_d_img, device)

                with torch.no_grad():
                    rgb_full = []
                    chunk = 8192
                    for i in range(0, rays_o_img.shape[0], chunk):
                        ro = rays_o_img[i:i+chunk]
                        rd = rays_d_img[i:i+chunk]
                        t_vals = torch.linspace(near, far, n_samples, device=device)
                        t_vals = t_vals.unsqueeze(0).expand(ro.shape[0], n_samples)
                        pts = ro.unsqueeze(1) + t_vals.unsqueeze(-1) * rd.unsqueeze(1)
                        rd_expand = rd.unsqueeze(1).expand(-1, n_samples, -1)
                        density, rgb = model(pts, rd_expand)
                        rgb_rendered = volrend(density, rgb, step_size)
                        rgb_full.append(rgb_rendered)
                    rgb_full = torch.cat(rgb_full, dim=0).reshape(H, W, 3).cpu().numpy()
                    plt.figure(figsize=(3,3))
                    plt.axis("off")
                    plt.imshow(np.clip(rgb_full, 0, 1))
                    plt.title(f"Iter {iteration+1}")
                    plt.savefig(f"progress_images/iter_{iteration+1}.png", bbox_inches='tight', pad_inches=0)
                    plt.close()

    return model, train_losses, val_psnrs, val_iterations


In [ ]:
model, losses, psnrs, iters = train_nerf_fast(
    images_train=images_train,
    c2ws_train=c2ws_train,
    images_val=images_val,
    c2ws_val=c2ws_val,
    K=K,
    num_iterations=2000,    # target
    batch_size=4096,
    n_samples=48,
    learning_rate=5e-4,
    device='cpu'
)


In [ ]:


def evaluate_on_validation(model, images_val, c2ws_val, K, H, W,
                           n_samples=64, near=2.0, far=6.0, device='cuda'):
    """
    Evaluate trained model on all validation images.
    """
    model.eval()
    
    step_size = (far - near) / n_samples
    val_psnrs = []
    
    print("Evaluating on validation set...")
    
    with torch.no_grad():
        for img_idx in tqdm(range(len(images_val))):
            # Create all rays for this validation image
            u = np.arange(W) + 0.5
            v = np.arange(H) + 0.5
            u_grid, v_grid = np.meshgrid(u, v)
            uv = np.stack([u_grid.flatten(), v_grid.flatten()], axis=-1)
            
            # Generate rays
            c2w_torch = torch.from_numpy(c2ws_val[img_idx]).float()
            K_torch = K if isinstance(K, torch.Tensor) else torch.from_numpy(K).float()
            uv_torch = torch.from_numpy(uv).float()
            
            rays_o, rays_d = pixel_to_ray(K_torch, c2w_torch, uv_torch)
            rays_o = rays_o.to(device)
            rays_d = rays_d.to(device)
            
            # Render in chunks
            chunk_size = 4096
            rgb_rendered_chunks = []
            
            for i in range(0, len(rays_o), chunk_size):
                rays_o_chunk = rays_o[i:i+chunk_size]
                rays_d_chunk = rays_d[i:i+chunk_size]
                
                # Sample points (no perturbation for validation)
                t_vals = torch.linspace(near, far, n_samples + 1, device=device)
                t_vals = (t_vals[:-1] + t_vals[1:]) / 2.0
                t_vals = t_vals.unsqueeze(0).expand(rays_o_chunk.shape[0], n_samples)
                points = rays_o_chunk.unsqueeze(1) + t_vals.unsqueeze(-1) * rays_d_chunk.unsqueeze(1)
                
                # Forward pass
                rays_d_expanded = rays_d_chunk.unsqueeze(1).expand(-1, n_samples, -1)
                density, rgb = model(points, rays_d_expanded)
                
                # Volume rendering
                rgb_rendered = volrend(density, rgb, step_size)
                rgb_rendered_chunks.append(rgb_rendered.cpu())
            
            # Concatenate chunks and reshape
            rgb_rendered_full = torch.cat(rgb_rendered_chunks, dim=0).numpy()
            rgb_rendered_img = rgb_rendered_full.reshape(H, W, 3)
            
            # Compute PSNR
            mse = np.mean((rgb_rendered_img - images_val[img_idx]) ** 2)
            psnr = -10 * np.log10(mse + 1e-8)
            val_psnrs.append(psnr)
            
            print(f"  Image {img_idx+1}/{len(images_val)}: PSNR = {psnr:.2f} dB")
    
    avg_psnr = np.mean(val_psnrs)
    print(f"\nAverage Validation PSNR: {avg_psnr:.2f} dB")
    
    return val_psnrs, avg_psnr

# Evaluate your trained model
val_psnrs_individual, avg_val_psnr = evaluate_on_validation(
    model=model,
    images_val=images_val,
    c2ws_val=c2ws_val,
    K=K,
    H=H,
    W=W,
    n_samples=64,
    near=2.0,
    far=6.0,
    device=device
)

print(f"\n{'='*70}")
print(f"FINAL VALIDATION PSNR: {avg_val_psnr:.2f} dB")
print(f"Target: 23 dB")
print(f"Status: {'✓ ACHIEVED' if avg_val_psnr >= 23 else '✗ NOT REACHED'}")
print(f"{'='*70}")

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(iters, psnrs, label="Validation PSNR", color='royalblue')
plt.xlabel("Iteration")
plt.ylabel("PSNR (dB)")
plt.title("Validation PSNR vs Iterations")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

from scipy.signal import savgol_filter

iters_arr = np.array(iters)
psnrs_arr = np.array(psnrs)

# window_length must be odd and ≤ len(psnrs)
window = min(11, len(psnrs_arr) if len(psnrs_arr)%2==1 else len(psnrs_arr)-1)
y_smooth = savgol_filter(psnrs_arr, window_length=window, polyorder=2)

plt.figure(figsize=(7,4))
plt.plot(iters_arr, y_smooth, color='royalblue', linewidth=2.0, label='PSNR (smoothed)')
plt.scatter(iters_arr, psnrs_arr, color='orange', s=25, label='Samples')  # optional
plt.xlabel("Iteration")
plt.ylabel("PSNR (dB)")
plt.title("Validation PSNR (Smoothed)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Check what keys are in your data file
data = np.load('lego_200x200.npz')
print("Keys in the npz file:")
print(list(data.keys()))
print()

# Show shapes of each array
for key in data.keys():
    print(f"{key}: {data[key].shape}")

In [ ]:
def render_spherical_views(
    model,
    data_path='lego_200x200.npz',
    output_path='lego_spherical_360.mp4',
    n_samples=64,
    near=2.0,
    far=6.0,
    device='cuda',
    fps=30,
    chunk_size=4096,
    save_frames=False
):
    """Render a spherical video using test camera poses from the npz file."""
    import imageio
    from pathlib import Path
    
    # Load data
    print("Loading data...")
    data = np.load(data_path)
    c2ws_test = data['c2ws_test']
    focal = float(data['focal'])  # Extract focal length
    
    # Get dimensions from training images
    H, W = data['images_train'].shape[1:3]
    
    # Construct K matrix from focal length and image dimensions
    K = np.array([
        [focal, 0, W/2],
        [0, focal, H/2],
        [0, 0, 1]
    ], dtype=np.float32)
    
    print(f"Image dimensions: {H}x{W}")
    print(f"Focal length: {focal}")
    print(f"Number of test views: {len(c2ws_test)}")
    print(f"Camera intrinsic matrix K:\n{K}")
    
    # Set model to eval
    model.eval()
    model.to(device)
    
    # Create output directory for frames if needed
    if save_frames:
        frames_dir = Path('spherical_frames')
        frames_dir.mkdir(exist_ok=True)
    
    # Render all views
    print(f"\nRendering {len(c2ws_test)} novel views...")
    frames = []
    
    for i, c2w in enumerate(tqdm(c2ws_test, desc="Rendering")):
        img = render_image(
            model=model,
            c2w=c2w,
            K=K,
            H=H,
            W=W,
            n_samples=n_samples,
            near=near,
            far=far,
            device=device,
            chunk_size=chunk_size
        )
        
        img_uint8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        frames.append(img_uint8)
        
        if save_frames:
            imageio.imwrite(f'spherical_frames/frame_{i:04d}.png', img_uint8)
    
    # Save video
    print(f"\nSaving video to {output_path}...")
    imageio.mimsave(output_path, frames, fps=fps)
    print(f"✓ Video saved!")
    
    # Show preview
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    indices = np.linspace(0, len(frames)-1, 8, dtype=int)
    for idx, ax in zip(indices, axes):
        ax.imshow(frames[idx])
        ax.set_title(f'Frame {idx}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    return frames

In [ ]:
# Load your current data
data = np.load('my_nerf_data.npz')

# Check what's in your file
print("Current keys in my_nerf_data.npz:")
print(list(data.keys()))
print()

# Show shapes
for key in data.keys():
    print(f"{key}: {data[key].shape}")

In [ ]:

def resize_nerf_dataset(
    input_path='my_nerf_data.npz',
    output_path='my_nerf_data_200x200.npz',
    target_size=(200, 200)
):
    """
    Resize all images in the dataset to a manageable size.
    
    Args:
        input_path: Original data file
        output_path: Where to save resized data
        target_size: (height, width) for resized images
    """
    
    print(f"Loading {input_path}...")
    data = np.load(input_path)
    
    H_target, W_target = target_size
    H_original, W_original = data['images_train'].shape[1:3]
    focal_original = float(data['focal'])
    
    print(f"Original image size: {H_original}x{W_original}")
    print(f"Target image size: {H_target}x{W_target}")
    print(f"Original focal length: {focal_original}")
    
    # Calculate new focal length (scales with image size)
    scale_h = H_target / H_original
    scale_w = W_target / W_original
    scale = (scale_h + scale_w) / 2  # Average scale
    focal_new = focal_original * scale
    
    print(f"New focal length: {focal_new}")
    
    # Resize training images
    print("\nResizing training images...")
    images_train_resized = []
    for i, img in enumerate(tqdm(data['images_train'], desc="Train")):
        img_resized = cv2.resize(img, (W_target, H_target), interpolation=cv2.INTER_AREA)
        images_train_resized.append(img_resized)
    images_train_resized = np.array(images_train_resized)
    
    # Resize validation images
    print("Resizing validation images...")
    images_val_resized = []
    for i, img in enumerate(tqdm(data['images_val'], desc="Val")):
        img_resized = cv2.resize(img, (W_target, H_target), interpolation=cv2.INTER_AREA)
        images_val_resized.append(img_resized)
    images_val_resized = np.array(images_val_resized)
    
    # Save resized data
    print(f"\nSaving to {output_path}...")
    np.savez(
        output_path,
        images_train=images_train_resized,
        images_val=images_val_resized,
        c2ws_train=data['c2ws_train'],
        c2ws_val=data['c2ws_val'],
        c2ws_test=data['c2ws_test'],
        focal=focal_new
    )
    
    print(f"\n✓ Resized dataset saved!")
    print(f"  images_train: {images_train_resized.shape}")
    print(f"  images_val: {images_val_resized.shape}")
    print(f"  focal: {focal_new}")
    
    return output_path

# Resize to 200x200 (good balance of quality and speed)
resized_path = resize_nerf_dataset(
    input_path='my_nerf_data.npz',
    output_path='my_nerf_data_200x200.npz',
    target_size=(200, 200)
)

In [ ]:

# Load the resized data
data = np.load('my_nerf_data_200x150.npz')

images_train = data['images_train']
images_val = data['images_val']
c2ws_train = data['c2ws_train']
c2ws_val = data['c2ws_val']
focal = float(data['focal'])

# Get image dimensions
H, W = images_train.shape[1:3]

# Build K matrix (camera intrinsics)
K = torch.tensor([
    [focal, 0, W/2],
    [0, focal, H/2],
    [0, 0, 1]
], dtype=torch.float32)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")
print(f"Image size: {H}x{W}")
print(f"Training images: {len(images_train)}")
print(f"Validation images: {len(images_val)}")
print(f"Focal length: {focal}")
print(f"\nIntrinsic matrix K:\n{K}")

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

def resize_nerf_dataset_proper(
    input_path='my_nerf_data.npz',
    output_path='my_nerf_data_200x150.npz',
    target_size= (200, 150)
):
    """
    Resize images AND properly scale camera intrinsics.
    This ensures the object stays in frame.
    """
    
    print(f"Loading {input_path}...")
    data = np.load(input_path)
    
    H_target, W_target = target_size
    H_original, W_original = data['images_train'].shape[1:3]
    focal_original = float(data['focal'])
    
    print(f"Original image size: {H_original}x{W_original}")
    print(f"Target image size: {H_target}x{W_target}")
    print(f"Original focal length: {focal_original}")
    
    # Calculate scale factors for height and width
    scale_h = H_target / H_original
    scale_w = W_target / W_original
    
    print(f"Height scale: {scale_h:.6f}")
    print(f"Width scale: {scale_w:.6f}")
    
    # Focal length should scale by the same factor
    # Use the average of both scales
    scale = (scale_h + scale_w) / 2.0
    focal_new = focal_original * scale
    
    print(f"New focal length: {focal_new:.6f}")
    
    # Verify the FOV is preserved
    fov_original_h = 2 * np.arctan(H_original / (2 * focal_original)) * 180 / np.pi
    fov_original_w = 2 * np.arctan(W_original / (2 * focal_original)) * 180 / np.pi
    fov_new_h = 2 * np.arctan(H_target / (2 * focal_new)) * 180 / np.pi
    fov_new_w = 2 * np.arctan(W_target / (2 * focal_new)) * 180 / np.pi
    
    print(f"\nField of View (should stay similar):")
    print(f"  Original: {fov_original_h:.2f}° (H) x {fov_original_w:.2f}° (W)")
    print(f"  New:      {fov_new_h:.2f}° (H) x {fov_new_w:.2f}° (W)")
    
    # Resize training images
    print("\nResizing training images...")
    images_train_resized = []
    for img in tqdm(data['images_train'], desc="Train"):
        # Ensure image is float32 in [0, 1]
        if img.dtype == np.uint8 or img.max() > 1.0:
            img = img.astype(np.float32) / 255.0
        
        # Use INTER_AREA for downsampling (best quality)
        img_resized = cv2.resize(img, (W_target, H_target), interpolation=cv2.INTER_AREA)
        images_train_resized.append(img_resized)
    
    images_train_resized = np.array(images_train_resized, dtype=np.float32)
    
    # Resize validation images
    print("Resizing validation images...")
    images_val_resized = []
    for img in tqdm(data['images_val'], desc="Val"):
        if img.dtype == np.uint8 or img.max() > 1.0:
            img = img.astype(np.float32) / 255.0
            
        img_resized = cv2.resize(img, (W_target, H_target), interpolation=cv2.INTER_AREA)
        images_val_resized.append(img_resized)
    
    images_val_resized = np.array(images_val_resized, dtype=np.float32)
    
    # Verify ranges
    print(f"\nFinal image ranges (should be [0, 1]):")
    print(f"  Train: [{images_train_resized.min():.4f}, {images_train_resized.max():.4f}]")
    print(f"  Val:   [{images_val_resized.min():.4f}, {images_val_resized.max():.4f}]")
    
    # Save with corrected focal length
    np.savez(
        output_path,
        images_train=images_train_resized,
        images_val=images_val_resized,
        c2ws_train=data['c2ws_train'],
        c2ws_val=data['c2ws_val'],
        c2ws_test=data['c2ws_test'],
        focal=focal_new  # Scaled focal length
    )
    
    print(f"\n✓ Dataset saved to: {output_path}")
    print(f"\nSummary:")
    print(f"  images_train: {images_train_resized.shape}")
    print(f"  images_val: {images_val_resized.shape}")
    print(f"  focal: {focal_new:.6f}")
    print(f"  FOV preserved: {abs(fov_new_h - fov_original_h) < 0.1}")
    
    return output_path

# Resize with proper intrinsics scaling
resized_path = resize_nerf_dataset_proper(
    input_path='my_nerf_data.npz',
    output_path='my_nerf_data_200x150.npz',
    target_size=(200, 150)
)

In [ ]:
import matplotlib.pyplot as plt

# Load original
data_original = np.load('my_nerf_data.npz')
img_original = data_original['images_train'][0]
if img_original.max() > 1.0:
    img_original = img_original / 255.0

# Load resized
data_resized = np.load('my_nerf_data_200x150.npz')
img_resized = data_resized['images_train'][0]

# Compare
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img_original)
axes[0].set_title(f'Original ({data_original["images_train"].shape[1]}x{data_original["images_train"].shape[2]})')
axes[0].axis('off')

axes[1].imshow(img_resized)
axes[1].set_title(f'Resized ({data_resized["images_train"].shape[1]}x{data_resized["images_train"].shape[2]})')
axes[1].axis('off')

plt.suptitle('Image Comparison - Object should be in same position', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print FOV info
focal_orig = float(data_original['focal'])
H_orig, W_orig = data_original['images_train'].shape[1:3]

focal_new = float(data_resized['focal'])
H_new, W_new = data_resized['images_train'].shape[1:3]

fov_orig = 2 * np.arctan(W_orig / (2 * focal_orig)) * 180 / np.pi
fov_new = 2 * np.arctan(W_new / (2 * focal_new)) * 180 / np.pi

print(f"Field of View comparison:")
print(f"  Original: {fov_orig:.2f}°")
print(f"  Resized:  {fov_new:.2f}°")
print(f"  Difference: {abs(fov_orig - fov_new):.4f}° (should be < 0.1°)")

In [ ]:
# Load the properly resized data
data = np.load('my_nerf_data_200x150.npz')

images_train = data['images_train']
images_val = data['images_val']
c2ws_train = data['c2ws_train']
c2ws_val = data['c2ws_val']
focal = float(data['focal'])

H, W = images_train.shape[1:3]

# Build K matrix with scaled focal length
K = torch.tensor([
    [focal, 0, W/2.0],
    [0, focal, H/2.0],
    [0, 0, 1]
], dtype=torch.float32)

print("Camera intrinsics (K matrix):")
print(K)
print(f"\nPrincipal point (should be center): ({W/2.0}, {H/2.0})")
print(f"Focal length: {focal:.6f}")

# Calculate FOV to verify
fov_horizontal = 2 * np.arctan(W / (2 * focal)) * 180 / np.pi
fov_vertical = 2 * np.arctan(H / (2 * focal)) * 180 / np.pi
print(f"\nField of View:")
print(f"  Horizontal: {fov_horizontal:.2f}°")
print(f"  Vertical: {fov_vertical:.2f}°")

In [ ]:
# Training parameters
num_iterations = 1500  # Adjust based on your needs
batch_size = 4096
n_samples = 64
learning_rate = 5e-4
near = 2.0  # Adjust based on your scene
far = 6.0   # Adjust based on your scene

# Train the model (using your existing train_nerf_fast function)
model, train_losses, val_psnrs, val_iterations, visualizations = train_nerf_fast(
    images_train=images_train,
    c2ws_train=c2ws_train,
    images_val=images_val,
    c2ws_val=c2ws_val,
    K=K,
    num_iterations=num_iterations,
    batch_size=batch_size,
    n_samples=n_samples,
    learning_rate=learning_rate,
    near=near,
    far=far,
    device=device
)

print("\n✓ Training complete!")

In [ ]:
import torch
import numpy as np

torch.serialization.add_safe_globals([np._core.multiarray.scalar])

ckpt = torch.load("checkpoints/nerf_iter1500.pth", map_location='cpu', weights_only=False)

train_losses = ckpt["train_losses"]
val_psnrs = ckpt["val_psnrs"]
val_iterations = ckpt["val_iterations"]

print(f"Loaded {len(train_losses)} train losses, {len(val_psnrs)} val PSNRs")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline

# Smooth PSNR curve
iters_arr = np.array(val_iterations)
psnrs_arr = np.array(val_psnrs)

x_smooth = np.linspace(iters_arr.min(), iters_arr.max(), 500)
spline = make_interp_spline(iters_arr, psnrs_arr, k=3)
y_smooth = spline(x_smooth)

plt.figure(figsize=(7,4))
plt.plot(x_smooth, y_smooth, color='royalblue', linewidth=2.5, label='PSNR (val, smoothed)')
plt.scatter(iters_arr, psnrs_arr, color='orange', s=30, label='Raw PSNR points')
plt.xlabel("Iteration")
plt.ylabel("PSNR (dB)")
plt.title("Validation PSNR Curve")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
# Training loss curve
plt.figure(figsize=(7,4))
plt.plot(range(len(train_losses)), train_losses, color='firebrick', linewidth=1.8)
plt.xlabel("Iteration")
plt.ylabel("Training Loss (MSE)")
plt.title("Training Loss Curve")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


unfortunately i was not able to render the frames for my own nerf, and thus could not make a gif